Работа с табличными данными

In [1]:
import pandas as pd
import numpy as np
import copy
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
from dotenv import load_dotenv
import wandb
import os
import logging
import time
import torch.nn as nn
import random
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import *
import pickle

In [2]:
df = pd.read_csv('dataset/train.csv', sep="|")
df

,trustLevel,totalScanTimeInSeconds,grandTotal,lineItemVoids,scansWithoutRegistration,quantityModifications,scannedLineItemsPerSecond,valuePerSecond,lineItemVoidsPerPosition,fraud
0,5,1054,54.70,7,0,3,0.027514,0.051898,0.241379,0
1,3,108,27.36,5,2,4,0.129630,0.253333,0.357143,0
2,3,1516,62.16,3,10,5,0.008575,0.041003,0.230769,0
3,6,1791,92.31,8,4,4,0.016192,0.051541,0.275862,0
4,5,430,81.53,3,7,2,0.062791,0.189605,0.111111,0
...,...,...,...,...,...,...,...,...,...,...
1874,1,321,76.03,8,7,2,0.071651,0.236854,0.347826,0
1875,1,397,41.89,5,5,0,0.065491,0.105516,0.192308,1
1876,4,316,41.83,5,8,1,0.094937,0.132373,0.166667,0
1877,2,685,62.68,1,6,2,0.035036,0.091504,0.041667,0


In [3]:
df.dtypes

trustLevel                     int64
totalScanTimeInSeconds         int64
grandTotal                   float64
lineItemVoids                  int64
scansWithoutRegistration       int64
quantityModifications          int64
scannedLineItemsPerSecond    float64
valuePerSecond               float64
lineItemVoidsPerPosition     float64
fraud                          int64
dtype: object

In [4]:
df.isna().sum()

trustLevel                   0
totalScanTimeInSeconds       0
grandTotal                   0
lineItemVoids                0
scansWithoutRegistration     0
quantityModifications        0
scannedLineItemsPerSecond    0
valuePerSecond               0
lineItemVoidsPerPosition     0
fraud                        0
dtype: int64

In [5]:
df.fraud.value_counts()

fraud
0    1775
1     104
Name: count, dtype: int64

In [6]:
y = df["fraud"]
X = df.drop(columns=["fraud"])

In [7]:
df.describe()

,trustLevel,totalScanTimeInSeconds,grandTotal,lineItemVoids,scansWithoutRegistration,quantityModifications,scannedLineItemsPerSecond,valuePerSecond,lineItemVoidsPerPosition,fraud
count,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000,1879.000000
mean,3.401809,932.153273,50.864492,5.469931,4.904204,2.525279,0.058138,0.201746,0.745404,0.055349
std,1.709404,530.144640,28.940202,3.451169,3.139697,1.695472,0.278512,1.242135,1.327241,0.228720
min,1.000000,2.000000,0.010000,0.000000,0.000000,0.000000,0.000548,0.000007,0.000000,0.000000
25%,2.000000,474.500000,25.965000,2.000000,2.000000,1.000000,0.008384,0.027787,0.160000,0.000000
50%,3.000000,932.000000,51.210000,5.000000,5.000000,3.000000,0.016317,0.054498,0.350000,0.000000
75%,5.000000,1397.000000,77.285000,8.000000,8.000000,4.000000,0.032594,0.107313,0.666667,0.000000
max,6.000000,1831.000000,99.960000,11.000000,10.000000,5.000000,6.666667,37.870000,11.000000,1.000000


## Предобработка данных

In [8]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [9]:
X_train.shape

(1503, 9)

In [10]:
X_val.shape

(376, 9)

In [11]:
X_test = pd.read_csv("dataset/test.csv", sep="|")
X_test

,trustLevel,totalScanTimeInSeconds,grandTotal,lineItemVoids,scansWithoutRegistration,quantityModifications,scannedLineItemsPerSecond,valuePerSecond,lineItemVoidsPerPosition
0,4,467,88.48,4,8,4,0.014989,0.189465,0.571429
1,3,1004,58.99,7,6,1,0.026892,0.058755,0.259259
2,1,162,14.00,4,5,4,0.006173,0.086420,4.000000
3,5,532,84.79,9,3,4,0.026316,0.159380,0.642857
4,5,890,42.16,4,0,0,0.021348,0.047371,0.210526
...,...,...,...,...,...,...,...,...,...
498116,4,783,59.10,2,2,0,0.012771,0.075479,0.200000
498117,1,278,98.90,9,5,4,0.050360,0.355755,0.642857
498118,3,300,5.41,6,6,4,0.030000,0.018033,0.666667
498119,2,1524,33.97,2,5,3,0.005906,0.022290,0.222222


In [12]:
y_test = pd.read_csv("dataset/DMC-2019-realclass.csv", sep="|")["fraud"]
y_test.value_counts()

fraud
0    474394
1     23727
Name: count, dtype: int64

In [13]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [14]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

In [15]:
y_train = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_val = torch.tensor(y_val.values, dtype=torch.float32).reshape(-1, 1)
y_test = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

In [16]:
BATCH_SIZE = 64

In [17]:
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=256)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=1024)

вес для редкого класса, чтобы модель его не пропускала

In [18]:
n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()

In [19]:
pos_weight = n_neg / n_pos
pos_weight

tensor(17.1084)

## Подготовка

In [20]:
load_dotenv()

WANDB_API_KEY = os.getenv("WANDB_API_KEY")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "gp5")
WANDB_ENTITY = os.getenv("WANDB_ENTITY")


In [21]:
wandb.login(key=WANDB_API_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/katya/.netrc
wandb: Currently logged in as: gigantina-ru (gigantina-ru-hse-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [22]:
def stop_logging():
    logger = logging.getLogger()
    for handler in logger.handlers:
        handler.flush()
        handler.close()
        logger.removeHandler(handler)

def new_log_file():
    stop_logging()
    timestamp = str(time.time()).replace('.', '_')
    log_file = f'part_2_{timestamp}.log'
    logging.basicConfig(
        filename=log_file,
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        force=True
    )
    logging.info("Начал логгировать новый запуск")
    return log_file

In [23]:
def evaluate_model(model, X, y, threshold=0.5, name="dataset"):
    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED) 

    model.eval()

    with torch.no_grad():
        logits = model(X)
        probs = torch.sigmoid(logits)

    y_true = y.numpy().ravel()
    y_prob = probs.numpy().ravel()
    y_pred = (y_prob > threshold).astype(int)

    roc_auc = roc_auc_score(y_true, y_prob)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f05 = fbeta_score(y_true, y_pred, beta=0.5)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    profit = tp * 5 - fp * 25 - fn * 5

    print(name)
    print("ROC-AUC:", round(roc_auc, 4))
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F0.5-score:", round(f05, 4))
    print("Profit:", profit)
    print()

    return {
        "dataset": name,
        "threshold": threshold,
        "roc_auc": roc_auc,
        "precision": precision,
        "recall": recall,
        "f05": f05,
        "profit": profit
    }

In [24]:
def train_model(model, train_loader, X_valid, y_valid, loss_fn, optimizer, 
    epochs=30, threshold=0.5,model_name="model"):
    
    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED) 

    best_metric = -10**9
    best_epoch = 0
    best_state = None

    history = []
    logging.info(f"Начали обучение {model_name}")

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0

        for xb, yb in train_loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        epoch_loss /= len(train_loader)

        valid_metrics = evaluate_model(model, X_valid, y_valid, threshold=threshold,
            name=f"{model_name} | valid epoch {epoch}")
            
        current_metric = valid_metrics["roc_auc"]
        
        if current_metric > best_metric:
            best_metric = current_metric
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

        print(f"Эпоха {epoch} | " f"Train Loss: {epoch_loss:.4f} | " 
        f"Val ROC-AUC: {valid_metrics['roc_auc']:.4f} | " f"Val Profit: {valid_metrics['profit']}")
        
        
        logging.info(
            f"Эпоха {epoch} | "
            f"Train Loss: {epoch_loss:.4f} | "
            f"Val ROC-AUC: {valid_metrics['roc_auc']:.4f} | "
            f"Val Profit: {valid_metrics['profit']}"
        )

        wandb.log({
             f"{model_name}/train_loss": epoch_loss,
             f"{model_name}/valid_profit": valid_metrics["profit"],
             f"{model_name}/valid_roc_auc": valid_metrics["roc_auc"],
             f"{model_name}/valid_precision": valid_metrics["precision"],
             f"{model_name}/valid_recall": valid_metrics["recall"],
             f"{model_name}/valid_f05": valid_metrics["f05"],
         })

    model.load_state_dict(best_state)

    print()
    print(f"Лучшая эпоха для {model_name}: {best_epoch}")
    print(f"Лучший ROC-AUC: {best_metric}")
    logging.info("Закончили обучение")
    logging.info(f"Лучшая эпоха для {model_name}: {best_epoch}")
    logging.info(f"Лучший ROC-AUC: {best_metric}")

    return model

In [25]:
def save_results(model, name, log_file, run):
    train_metrics = evaluate_model(model, X_train, y_train, threshold=0.5, name="Train")
    test_metrics = evaluate_model(model, X_test, y_test, threshold=0.5, name="Test")
    pickle.dump(model.state_dict(), open(f"models/{name}.pkl", 'wb'))
    logging.info("Сохранили веса модели в папку models")

    metric_keys = list(train_metrics.keys())

    table = wandb.Table(columns=metric_keys)
    table.add_data(*[train_metrics[i] for i in metric_keys])
    table.add_data(*[test_metrics[i] for i in metric_keys])
    wandb.log({'results': table})
    artifact = wandb.Artifact(name=name, type="model", description=f"Тест логгирования модели: {name}")

    artifact.add_file(f"models/{name}.pkl")
    artifact.add_file(log_file)
    run.log_artifact(artifact) 

## model_1_baseline

In [48]:
EPOCHS = 30
SEED = 42

Отключаем логирование на переборе гиперпараметров

In [49]:
wandb.init(mode="disabled")
results = []
for LR in [0.05, 0.03, 0.01, 0.005, 0.002, 0.0001]:
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED) 

        model_1 = nn.Sequential(
            nn.Linear(9, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
        )

        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(model_1.parameters(), lr=LR)

        model_1 = train_model(
            model=model_1,
            train_loader=train_loader,
            X_valid=X_val,
            y_valid=y_val,
            loss_fn=loss_fn,
            optimizer=optimizer,
            epochs=EPOCHS,
            threshold=0.5,
            model_name="model_1_baseline"
        )

        val = evaluate_model(model_1, X_val, y_val, threshold=0.5, name="val")
        results.append({"lr": LR, "profit": val["profit"], "roc_auc": val["roc_auc"]})

model_1_baseline | valid epoch 1
ROC-AUC: 0.9363
Precision: 0.2769
Recall: 0.8571
F0.5-score: 0.3203
Profit: -1100

Эпоха 1 | Train Loss: 0.8771 | Val ROC-AUC: 0.9363 | Val Profit: -1100
model_1_baseline | valid epoch 2
ROC-AUC: 0.9608
Precision: 0.2019
Recall: 1.0
F0.5-score: 0.2403
Profit: -1970

Эпоха 2 | Train Loss: 0.6035 | Val ROC-AUC: 0.9608 | Val Profit: -1970
model_1_baseline | valid epoch 3
ROC-AUC: 0.972
Precision: 0.2283
Recall: 1.0
F0.5-score: 0.2699
Profit: -1670

Эпоха 3 | Train Loss: 0.4833 | Val ROC-AUC: 0.9720 | Val Profit: -1670
model_1_baseline | valid epoch 4
ROC-AUC: 0.9714
Precision: 0.3281
Recall: 1.0
F0.5-score: 0.3791
Profit: -970

Эпоха 4 | Train Loss: 0.3636 | Val ROC-AUC: 0.9714 | Val Profit: -970
model_1_baseline | valid epoch 5
ROC-AUC: 0.9771
Precision: 0.3443
Recall: 1.0
F0.5-score: 0.3962
Profit: -895

Эпоха 5 | Train Loss: 0.3314 | Val ROC-AUC: 0.9771 | Val Profit: -895
model_1_baseline | valid epoch 6
ROC-AUC: 0.9685
Precision: 0.3387
Recall: 1.0
F0.

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_1_baseline | valid epoch 3
ROC-AUC: 0.9548
Precision: 0.2333
Recall: 1.0
F0.5-score: 0.2756
Profit: -1620

Эпоха 3 | Train Loss: 0.7432 | Val ROC-AUC: 0.9548 | Val Profit: -1620
model_1_baseline | valid epoch 4
ROC-AUC: 0.9651
Precision: 0.236
Recall: 1.0
F0.5-score: 0.2785
Profit: -1595

Эпоха 4 | Train Loss: 0.4980 | Val ROC-AUC: 0.9651 | Val Profit: -1595
model_1_baseline | valid epoch 5
ROC-AUC: 0.9671
Precision: 0.2471
Recall: 1.0
F0.5-score: 0.2909
Profit: -1495

Эпоха 5 | Train Loss: 0.4175 | Val ROC-AUC: 0.9671 | Val Profit: -1495
model_1_baseline | valid epoch 6
ROC-AUC: 0.9677
Precision: 0.2658
Recall: 1.0
F0.5-score: 0.3116
Profit: -1345

Эпоха 6 | Train Loss: 0.3743 | Val ROC-AUC: 0.9677 | Val Profit: -1345
model_1_baseline | valid epoch 7
ROC-AUC: 0.97
Precision: 0.28
Recall: 1.0
F0.5-score: 0.3271
Profit: -1245

Эпоха 7 | Train Loss: 0.3403 | Val ROC-AUC: 0.9700 | Val Profit: -1245
model_1_baseline | valid epoch 8
ROC-AUC: 0.9724
Precision: 0.2877
Recall: 1.0
F0.5-s

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

Эпоха 10 | Train Loss: 0.4160 | Val ROC-AUC: 0.9677 | Val Profit: -1420
model_1_baseline | valid epoch 11
ROC-AUC: 0.9693
Precision: 0.2727
Recall: 1.0
F0.5-score: 0.3191
Profit: -1295

Эпоха 11 | Train Loss: 0.3885 | Val ROC-AUC: 0.9693 | Val Profit: -1295
model_1_baseline | valid epoch 12
ROC-AUC: 0.9702
Precision: 0.2917
Recall: 1.0
F0.5-score: 0.3398
Profit: -1170

Эпоха 12 | Train Loss: 0.3652 | Val ROC-AUC: 0.9702 | Val Profit: -1170
model_1_baseline | valid epoch 13
ROC-AUC: 0.9704
Precision: 0.2958
Recall: 1.0
F0.5-score: 0.3443
Profit: -1145

Эпоха 13 | Train Loss: 0.3463 | Val ROC-AUC: 0.9704 | Val Profit: -1145
model_1_baseline | valid epoch 14
ROC-AUC: 0.9712
Precision: 0.3
Recall: 1.0
F0.5-score: 0.3488
Profit: -1120

Эпоха 14 | Train Loss: 0.3299 | Val ROC-AUC: 0.9712 | Val Profit: -1120
model_1_baseline | valid epoch 15
ROC-AUC: 0.9728
Precision: 0.3043
Recall: 1.0
F0.5-score: 0.3535
Profit: -1095

Эпоха 15 | Train Loss: 0.3154 | Val ROC-AUC: 0.9728 | Val Profit: -1095
m

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

model_1_baseline | valid epoch 7
ROC-AUC: 0.7803
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 7 | Train Loss: 1.3232 | Val ROC-AUC: 0.7803 | Val Profit: -105
model_1_baseline | valid epoch 8
ROC-AUC: 0.7918
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 8 | Train Loss: 1.3215 | Val ROC-AUC: 0.7918 | Val Profit: -105
model_1_baseline | valid epoch 9
ROC-AUC: 0.8013
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 9 | Train Loss: 1.3198 | Val ROC-AUC: 0.8013 | Val Profit: -105
model_1_baseline | valid epoch 10
ROC-AUC: 0.8097
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 10 | Train Loss: 1.3181 | Val ROC-AUC: 0.8097 | Val Profit: -105
model_1_baseline | valid epoch 11
ROC-AUC: 0.8193
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 11 | Train Loss: 1.3163 | Val ROC-AUC: 0.8193 | Val Profit: -105
model_1_baseline | valid epoch 12
ROC-AUC: 0.8284
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 12 

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

model_1_baseline | valid epoch 18
ROC-AUC: 0.8565
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 18 | Train Loss: 1.3029 | Val ROC-AUC: 0.8565 | Val Profit: -105
model_1_baseline | valid epoch 19
ROC-AUC: 0.8596
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 19 | Train Loss: 1.3007 | Val ROC-AUC: 0.8596 | Val Profit: -105
model_1_baseline | valid epoch 20
ROC-AUC: 0.8633
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 20 | Train Loss: 1.2985 | Val ROC-AUC: 0.8633 | Val Profit: -105
model_1_baseline | valid epoch 21
ROC-AUC: 0.8611
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 21 | Train Loss: 1.2962 | Val ROC-AUC: 0.8611 | Val Profit: -105
model_1_baseline | valid epoch 22
ROC-AUC: 0.8636
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 22 | Train Loss: 1.2938 | Val ROC-AUC: 0.8636 | Val Profit: -105
model_1_baseline | valid epoch 23
ROC-AUC: 0.8691
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпо

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

In [50]:
best = max(results, key=lambda r: r["profit"])
print("Лучшие гиперпараметры:", best)

Лучшие гиперпараметры: {'lr': 0.0001, 'profit': np.int64(-105), 'roc_auc': 0.8847753185781355}


In [51]:
LR = best["lr"]

In [52]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 

model_1 = nn.Sequential(
    nn.Linear(9, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_1.parameters(), lr=LR)
config = {
    "model": "MLP",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_1)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_1_baseline", config=config)
log_file = new_log_file()

model_1 = train_model(
    model=model_1,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_1_baseline"
)

save_results(model_1, "model_1_baseline", log_file, run)

run.finish()

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

model_1_baseline | valid epoch 1
ROC-AUC: 0.6702
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 1 | Train Loss: 1.3347 | Val ROC-AUC: 0.6702 | Val Profit: -105
model_1_baseline | valid epoch 2
ROC-AUC: 0.6936
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 2 | Train Loss: 1.3326 | Val ROC-AUC: 0.6936 | Val Profit: -105
model_1_baseline | valid epoch 3
ROC-AUC: 0.7123
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 3 | Train Loss: 1.3305 | Val ROC-AUC: 0.7123 | Val Profit: -105
model_1_baseline | valid epoch 4
ROC-AUC: 0.7319
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 4 | Train Loss: 1.3286 | Val ROC-AUC: 0.7319 | Val Profit: -105
model_1_baseline | valid epoch 5
ROC-AUC: 0.7497
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 5 | Train Loss: 1.3267 | Val ROC-AUC: 0.7497 | Val Profit: -105
model_1_baseline | valid epoch 6
ROC-AUC: 0.7667
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 6 | Trai

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

model_1_baseline | valid epoch 14
ROC-AUC: 0.8406
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 14 | Train Loss: 1.3109 | Val ROC-AUC: 0.8406 | Val Profit: -105
model_1_baseline | valid epoch 15
ROC-AUC: 0.8451
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 15 | Train Loss: 1.3090 | Val ROC-AUC: 0.8451 | Val Profit: -105
model_1_baseline | valid epoch 16
ROC-AUC: 0.8492
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 16 | Train Loss: 1.3070 | Val ROC-AUC: 0.8492 | Val Profit: -105
model_1_baseline | valid epoch 17
ROC-AUC: 0.8534
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 17 | Train Loss: 1.3050 | Val ROC-AUC: 0.8534 | Val Profit: -105
model_1_baseline | valid epoch 18
ROC-AUC: 0.8565
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 18 | Train Loss: 1.3029 | Val ROC-AUC: 0.8565 | Val Profit: -105
model_1_baseline | valid epoch 19
ROC-AUC: 0.8596
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпо

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

model_1_baseline | valid epoch 23
ROC-AUC: 0.8691
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 23 | Train Loss: 1.2913 | Val ROC-AUC: 0.8691 | Val Profit: -105
model_1_baseline | valid epoch 24
ROC-AUC: 0.8719
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 24 | Train Loss: 1.2887 | Val ROC-AUC: 0.8719 | Val Profit: -105
model_1_baseline | valid epoch 25
ROC-AUC: 0.8735
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 25 | Train Loss: 1.2860 | Val ROC-AUC: 0.8735 | Val Profit: -105
model_1_baseline | valid epoch 26
ROC-AUC: 0.8771
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 26 | Train Loss: 1.2832 | Val ROC-AUC: 0.8771 | Val Profit: -105
model_1_baseline | valid epoch 27
ROC-AUC: 0.8794
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 27 | Train Loss: 1.2802 | Val ROC-AUC: 0.8794 | Val Profit: -105
model_1_baseline | valid epoch 28
ROC-AUC: 0.8818
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпо

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Test
ROC-AUC: 0.9032
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -118635



model_1_baseline/train_loss,███▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▂▂▂▁▁
model_1_baseline/valid_f05,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
model_1_baseline/valid_precision,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
model_1_baseline/valid_profit,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
model_1_baseline/valid_recall,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
model_1_baseline/valid_roc_auc,▁▂▂▃▄▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇███████
model_1_baseline/train_loss,1.27086
model_1_baseline/valid_f05,0
model_1_baseline/valid_precision,0
model_1_baseline/valid_profit,-105
model_1_baseline/valid_recall,0


In [53]:
train_metrics = evaluate_model(model_1, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_1, X_test, y_test, threshold=0.5, name="Test")

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Train
ROC-AUC: 0.9065
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -415

Test
ROC-AUC: 0.9032
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -118635



Видно предупреждение в выводе о том, что модель не предсказала ни одного фрода. Получилась недообученная модель. Исходя из функции profit был выбран самый маленький lr, и модель с ним не успела обучиться

### Попробуем подобрать порог

In [34]:
with torch.no_grad():
    val_prob = torch.sigmoid(model(X_val)).numpy()

val_true = y_val.numpy()

NameError: name 'model' is not defined

In [ ]:
from sklearn.metrics import precision_recall_curve

target_recall = 0.90

prec, rec, thr = precision_recall_curve(val_true, val_prob)
best_threshold = thr[rec[:-1] >= target_recall].max()
logging.info(f"Подобрали порог {best_threshold}")
wandb.log({"best_threshold": best_threshold})
best_threshold

In [ ]:
y_pred = (y_prob > best_threshold).astype(int)

In [ ]:
pre_at_score = precision_score(y_true, y_pred)
#logging.info(f"На тесте Precision: {pre_at_score}")
pre_at_score

In [ ]:
rec_at_score = recall_score(y_true, y_pred)
#logging.info(f"На тесте Recall: {rec_at_score}")
rec_at_score

In [ ]:
import matplotlib.pyplot as plt

plt.plot(rec, prec)
plt.xlabel("recall")
plt.ylabel("precision")
plt.title("PR кривая")
plt.show()

In [ ]:
#import pickle

#pickle.dump(model.state_dict(), open("models/model_fraud_1.pkl", 'wb'))
#logging.info("Сохранили веса модели в папку models")

In [ ]:
#results = wandb.Table(columns=['model', 'test/roc_auc', 'test/precision', 'test/recall', 'test/recall_new_threshold', 'test/precision_new_threshold'])
#results.add_data("nn_baseline", auc_score, pre_score, rec_score, rec_at_score, pre_at_score)
#wandb.log({"results": results})

In [ ]:
#artifact = wandb.Artifact(name="model_fraud_1", type="model", description="Тест логгирования модели MLP")

#artifact.add_file("models/model_fraud_1.pkl")
#artifact.add_file(log_file)
#run.log_artifact(artifact)




In [ ]:
#run.finish()

## model_2_dop_sloi

In [54]:
EPOCHS = 30
SEED = 42

In [55]:
wandb.init(mode="disabled")
results = []
for LR in [0.05, 0.03, 0.01, 0.005, 0.002, 0.0001]:
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED) 

        model_2 = nn.Sequential(
            nn.Linear(9, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 8),
            nn.ReLU(),

            nn.Linear(8, 1)
        )

        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(model_2.parameters(), lr=LR)

        model_2 = train_model(
            model=model_2,
            train_loader=train_loader,
            X_valid=X_val,
            y_valid=y_val,
            loss_fn=loss_fn,
            optimizer=optimizer,
            epochs=EPOCHS,
            threshold=0.5,
            model_name="model_2_dop_sloi"
        )

        val = evaluate_model(model_2, X_val, y_val, threshold=0.5, name="val")
        results.append({"lr": LR, "profit": val["profit"], "roc_auc": val["roc_auc"]})

model_2_dop_sloi | valid epoch 1
ROC-AUC: 0.9133
Precision: 0.2346
Recall: 0.9048
F0.5-score: 0.2754
Profit: -1465

Эпоха 1 | Train Loss: 1.0974 | Val ROC-AUC: 0.9133 | Val Profit: -1465
model_2_dop_sloi | valid epoch 2
ROC-AUC: 0.9355
Precision: 0.2879
Recall: 0.9048
F0.5-score: 0.3333
Profit: -1090

Эпоха 2 | Train Loss: 0.6756 | Val ROC-AUC: 0.9355 | Val Profit: -1090
model_2_dop_sloi | valid epoch 3
ROC-AUC: 0.9292
Precision: 0.2471
Recall: 1.0
F0.5-score: 0.2909
Profit: -1495

Эпоха 3 | Train Loss: 0.5462 | Val ROC-AUC: 0.9292 | Val Profit: -1495
model_2_dop_sloi | valid epoch 4
ROC-AUC: 0.9064
Precision: 0.2211
Recall: 1.0
F0.5-score: 0.2618
Profit: -1745

Эпоха 4 | Train Loss: 0.4843 | Val ROC-AUC: 0.9064 | Val Profit: -1745
model_2_dop_sloi | valid epoch 5
ROC-AUC: 0.862
Precision: 0.1544
Recall: 1.0
F0.5-score: 0.1858
Profit: -2770

Эпоха 5 | Train Loss: 0.5720 | Val ROC-AUC: 0.8620 | Val Profit: -2770
model_2_dop_sloi | valid epoch 6
ROC-AUC: 0.869
Precision: 0.1556
Recall: 1

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Эпоха 4 | Train Loss: 0.4091 | Val ROC-AUC: 0.9599 | Val Profit: -1395
model_2_dop_sloi | valid epoch 5
ROC-AUC: 0.9616
Precision: 0.2727
Recall: 1.0
F0.5-score: 0.3191
Profit: -1295

Эпоха 5 | Train Loss: 0.3485 | Val ROC-AUC: 0.9616 | Val Profit: -1295
model_2_dop_sloi | valid epoch 6
ROC-AUC: 0.9544
Precision: 0.2763
Recall: 1.0
F0.5-score: 0.3231
Profit: -1270

Эпоха 6 | Train Loss: 0.3119 | Val ROC-AUC: 0.9544 | Val Profit: -1270
model_2_dop_sloi | valid epoch 7
ROC-AUC: 0.9531
Precision: 0.2658
Recall: 1.0
F0.5-score: 0.3116
Profit: -1345

Эпоха 7 | Train Loss: 0.2901 | Val ROC-AUC: 0.9531 | Val Profit: -1345
model_2_dop_sloi | valid epoch 8
ROC-AUC: 0.9612
Precision: 0.3
Recall: 1.0
F0.5-score: 0.3488
Profit: -1120

Эпоха 8 | Train Loss: 0.2719 | Val ROC-AUC: 0.9612 | Val Profit: -1120
model_2_dop_sloi | valid epoch 9
ROC-AUC: 0.9646
Precision: 0.3571
Recall: 0.9524
F0.5-score: 0.4082
Profit: -805

Эпоха 9 | Train Loss: 0.2528 | Val ROC-AUC: 0.9646 | Val Profit: -805
model_2_dop

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_2_dop_sloi | valid epoch 10
ROC-AUC: 0.9643
Precision: 0.3231
Recall: 1.0
F0.5-score: 0.3737
Profit: -995

Эпоха 10 | Train Loss: 0.2824 | Val ROC-AUC: 0.9643 | Val Profit: -995
model_2_dop_sloi | valid epoch 11
ROC-AUC: 0.9654
Precision: 0.3443
Recall: 1.0
F0.5-score: 0.3962
Profit: -895

Эпоха 11 | Train Loss: 0.2611 | Val ROC-AUC: 0.9654 | Val Profit: -895
model_2_dop_sloi | valid epoch 12
ROC-AUC: 0.9666
Precision: 0.3509
Recall: 0.9524
F0.5-score: 0.4016
Profit: -830

Эпоха 12 | Train Loss: 0.2427 | Val ROC-AUC: 0.9666 | Val Profit: -830
model_2_dop_sloi | valid epoch 13
ROC-AUC: 0.9678
Precision: 0.3585
Recall: 0.9048
F0.5-score: 0.4077
Profit: -765

Эпоха 13 | Train Loss: 0.2258 | Val ROC-AUC: 0.9678 | Val Profit: -765
model_2_dop_sloi | valid epoch 14
ROC-AUC: 0.9689
Precision: 0.3654
Recall: 0.9048
F0.5-score: 0.4148
Profit: -740

Эпоха 14 | Train Loss: 0.2121 | Val ROC-AUC: 0.9689 | Val Profit: -740
model_2_dop_sloi | valid epoch 15
ROC-AUC: 0.969
Precision: 0.3725
Reca

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

model_2_dop_sloi | valid epoch 5
ROC-AUC: 0.7105
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 5 | Train Loss: 1.3061 | Val ROC-AUC: 0.7105 | Val Profit: -105
model_2_dop_sloi | valid epoch 6
ROC-AUC: 0.7498
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 6 | Train Loss: 1.3044 | Val ROC-AUC: 0.7498 | Val Profit: -105
model_2_dop_sloi | valid epoch 7
ROC-AUC: 0.7851
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 7 | Train Loss: 1.3024 | Val ROC-AUC: 0.7851 | Val Profit: -105
model_2_dop_sloi | valid epoch 8
ROC-AUC: 0.8165
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 8 | Train Loss: 1.3002 | Val ROC-AUC: 0.8165 | Val Profit: -105
model_2_dop_sloi | valid epoch 9
ROC-AUC: 0.8377
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 9 | Train Loss: 1.2976 | Val ROC-AUC: 0.8377 | Val Profit: -105
model_2_dop_sloi | valid epoch 10
ROC-AUC: 0.8541
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 10 | Tr

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_2_dop_sloi | valid epoch 18
ROC-AUC: 0.9302
Precision: 0.4333
Recall: 0.619
F0.5-score: 0.461
Profit: -400

Эпоха 18 | Train Loss: 1.2529 | Val ROC-AUC: 0.9302 | Val Profit: -400
model_2_dop_sloi | valid epoch 19
ROC-AUC: 0.9341
Precision: 0.3659
Recall: 0.7143
F0.5-score: 0.4054
Profit: -605

Эпоха 19 | Train Loss: 1.2445 | Val ROC-AUC: 0.9341 | Val Profit: -605
model_2_dop_sloi | valid epoch 20
ROC-AUC: 0.9374
Precision: 0.3462
Recall: 0.8571
F0.5-score: 0.393
Profit: -775

Эпоха 20 | Train Loss: 1.2353 | Val ROC-AUC: 0.9374 | Val Profit: -775
model_2_dop_sloi | valid epoch 21
ROC-AUC: 0.9398
Precision: 0.3065
Recall: 0.9048
F0.5-score: 0.3532
Profit: -990

Эпоха 21 | Train Loss: 1.2251 | Val ROC-AUC: 0.9398 | Val Profit: -990
model_2_dop_sloi | valid epoch 22
ROC-AUC: 0.9427
Precision: 0.2923
Recall: 0.9048
F0.5-score: 0.3381
Profit: -1065

Эпоха 22 | Train Loss: 1.2141 | Val ROC-AUC: 0.9427 | Val Profit: -1065
model_2_dop_sloi | valid epoch 23
ROC-AUC: 0.9453
Precision: 0.267

In [56]:
best = max(results, key=lambda r: r["profit"])
print("Лучшие гиперпараметры:", best)

Лучшие гиперпараметры: {'lr': 0.01, 'profit': np.int64(-185), 'roc_auc': 0.9884641180415827}


In [57]:
LR = best["lr"]

In [58]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_2 = nn.Sequential(
    nn.Linear(9, 32),
    nn.ReLU(),

    nn.Linear(32, 16),
    nn.ReLU(),

    nn.Linear(16, 8),
    nn.ReLU(),

    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_2.parameters(), lr=LR)

config = {
    "model": "MLP_added_layer",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_2)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_2_added_layers", config=config)
log_file = new_log_file()

model_2 = train_model(
    model=model_2,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_2_dop_sloi"
)

save_results(model_2, "model_2", log_file, run)

run.finish()

model_2_dop_sloi | valid epoch 1
ROC-AUC: 0.9311
Precision: 0.3051
Recall: 0.8571
F0.5-score: 0.3502
Profit: -950

Эпоха 1 | Train Loss: 1.1825 | Val ROC-AUC: 0.9311 | Val Profit: -950
model_2_dop_sloi | valid epoch 2
ROC-AUC: 0.9552
Precision: 0.3
Recall: 1.0
F0.5-score: 0.3488
Profit: -1120

Эпоха 2 | Train Loss: 0.6270 | Val ROC-AUC: 0.9552 | Val Profit: -1120
model_2_dop_sloi | valid epoch 3
ROC-AUC: 0.9502
Precision: 0.25
Recall: 1.0
F0.5-score: 0.2941
Profit: -1470

Эпоха 3 | Train Loss: 0.4225 | Val ROC-AUC: 0.9502 | Val Profit: -1470
model_2_dop_sloi | valid epoch 4
ROC-AUC: 0.9642
Precision: 0.2857
Recall: 0.9524
F0.5-score: 0.3322
Profit: -1155

Эпоха 4 | Train Loss: 0.4105 | Val ROC-AUC: 0.9642 | Val Profit: -1155
model_2_dop_sloi | valid epoch 5
ROC-AUC: 0.9641
Precision: 0.3182
Recall: 1.0
F0.5-score: 0.3684
Profit: -1020

Эпоха 5 | Train Loss: 0.3026 | Val ROC-AUC: 0.9641 | Val Profit: -1020
model_2_dop_sloi | valid epoch 6
ROC-AUC: 0.9599
Precision: 0.2941
Recall: 0.9524

model_2_dop_sloi/train_loss,█▄▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▂▁▁▃▂▁▁▁▁▁▃▂
model_2_dop_sloi/valid_f05,▂▂▁▂▂▂▃▃▄▃▄▄▅▅▅▆▆▇▇▄▆▂▄▅▇█▇█▂▄
model_2_dop_sloi/valid_precision,▂▂▁▂▂▂▃▃▄▃▃▃▄▄▄▆▆▇▇▄▆▂▄▅▇█▆█▂▄
model_2_dop_sloi/valid_profit,▄▃▁▃▃▃▄▅▆▅▅▅▆▆▆▇▇▇█▅▇▃▆▆▇█▇█▃▆
model_2_dop_sloi/valid_recall,▄██▇█▇▇▇▁█▅█▇▄▅▅▅▅▄█▂███▇▂▇▄██
model_2_dop_sloi/valid_roc_auc,▁▄▃▅▅▅▅▅▅▅▅▆▇▆▆▇▇▇▇▆▆▅▆████▇▆▇
model_2_dop_sloi/train_loss,0.30533
model_2_dop_sloi/valid_f05,0.47511
model_2_dop_sloi/valid_precision,0.42
model_2_dop_sloi/valid_profit,-620
model_2_dop_sloi/valid_recall,1


In [59]:
train_metrics = evaluate_model(model_2, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_2, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9974
Precision: 0.7757
Recall: 1.0
F0.5-score: 0.8121
Profit: -185

Test
ROC-AUC: 0.9858
Precision: 0.6243
Recall: 0.8244
F0.5-score: 0.6561
Profit: -217335



Да, добавив всего 1 доп слой, видно, что модель стала находить зависимости. Precision и Recall больше не равны нулю. Profit равен -217335 (что однако хуже бейзлайна без предсказания фрода)

## model_3_bolshe_neyronov

In [61]:
EPOCHS = 30
SEED = 42

In [62]:
wandb.init(mode="disabled")
results = []
for LR in [0.05, 0.03, 0.01, 0.005, 0.002, 0.0001]:
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED) 

        model_3 = nn.Sequential(
            nn.Linear(9, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(model_3.parameters(), lr=LR)

        model_3 = train_model(
            model=model_3,
            train_loader=train_loader,
            X_valid=X_val,
            y_valid=y_val,
            loss_fn=loss_fn,
            optimizer=optimizer,
            epochs=EPOCHS,
            threshold=0.5,
            model_name="model_3_bolshe_neyronov"
        )

        val = evaluate_model(model_3, X_val, y_val, threshold=0.5, name="val")
        results.append({"lr": LR, "profit": val["profit"], "roc_auc": val["roc_auc"]})

model_3_bolshe_neyronov | valid epoch 1
ROC-AUC: 0.9429
Precision: 0.2727
Recall: 0.8571
F0.5-score: 0.3158
Profit: -1125

Эпоха 1 | Train Loss: 0.8665 | Val ROC-AUC: 0.9429 | Val Profit: -1125
model_3_bolshe_neyronov | valid epoch 2
ROC-AUC: 0.9569
Precision: 0.253
Recall: 1.0
F0.5-score: 0.2975
Profit: -1445

Эпоха 2 | Train Loss: 0.5109 | Val ROC-AUC: 0.9569 | Val Profit: -1445
model_3_bolshe_neyronov | valid epoch 3
ROC-AUC: 0.9693
Precision: 0.3509
Recall: 0.9524
F0.5-score: 0.4016
Profit: -830

Эпоха 3 | Train Loss: 0.4088 | Val ROC-AUC: 0.9693 | Val Profit: -830
model_3_bolshe_neyronov | valid epoch 4
ROC-AUC: 0.9698
Precision: 0.3922
Recall: 0.9524
F0.5-score: 0.4444
Profit: -680

Эпоха 4 | Train Loss: 0.3122 | Val ROC-AUC: 0.9698 | Val Profit: -680
model_3_bolshe_neyronov | valid epoch 5
ROC-AUC: 0.9689
Precision: 0.3443
Recall: 1.0
F0.5-score: 0.3962
Profit: -895

Эпоха 5 | Train Loss: 0.2802 | Val ROC-AUC: 0.9689 | Val Profit: -895
model_3_bolshe_neyronov | valid epoch 6
ROC

In [63]:
best = max(results, key=lambda r: r["profit"])
print("Лучшие гиперпараметры:", best)

Лучшие гиперпараметры: {'lr': 0.005, 'profit': np.int64(-150), 'roc_auc': 0.984842387659289}


In [64]:
LR = best["lr"]

In [65]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 

model_3 = nn.Sequential(
    nn.Linear(9, 32),
    nn.ReLU(),
    nn.Linear(32, 64),
    nn.ReLU(),
    nn.Linear(64, 1),
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_3.parameters(), lr=LR)

config = {
    "model": "MLP_added_neurons",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_3)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_3_more_neurons", config=config)
log_file = new_log_file()

model_3 = train_model(
    model=model_3,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_3_bolshe_neyronov"
)


save_results(model_3, "model_3", log_file, run)

run.finish()

model_3_bolshe_neyronov | valid epoch 1
ROC-AUC: 0.9571
Precision: 0.1615
Recall: 1.0
F0.5-score: 0.1941
Profit: -2620

Эпоха 1 | Train Loss: 1.0990 | Val ROC-AUC: 0.9571 | Val Profit: -2620
model_3_bolshe_neyronov | valid epoch 2
ROC-AUC: 0.9622
Precision: 0.2234
Recall: 1.0
F0.5-score: 0.2645
Profit: -1720

Эпоха 2 | Train Loss: 0.5891 | Val ROC-AUC: 0.9622 | Val Profit: -1720
model_3_bolshe_neyronov | valid epoch 3
ROC-AUC: 0.9622
Precision: 0.2658
Recall: 1.0
F0.5-score: 0.3116
Profit: -1345

Эпоха 3 | Train Loss: 0.4419 | Val ROC-AUC: 0.9622 | Val Profit: -1345
model_3_bolshe_neyronov | valid epoch 4
ROC-AUC: 0.9653
Precision: 0.2561
Recall: 1.0
F0.5-score: 0.3009
Profit: -1420

Эпоха 4 | Train Loss: 0.3847 | Val ROC-AUC: 0.9653 | Val Profit: -1420
model_3_bolshe_neyronov | valid epoch 5
ROC-AUC: 0.9674
Precision: 0.2692
Recall: 1.0
F0.5-score: 0.3153
Profit: -1320

Эпоха 5 | Train Loss: 0.3440 | Val ROC-AUC: 0.9674 | Val Profit: -1320
model_3_bolshe_neyronov | valid epoch 6
ROC-A

model_3_bolshe_neyronov/train_loss,█▅▄▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
model_3_bolshe_neyronov/valid_f05,▁▂▃▂▃▃▃▃▄▄▄▅▅▅▅▅▆▆▇▇█▇▇▇█▆▇▆▆▇
model_3_bolshe_neyronov/valid_precision,▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▅▆▆▇▇█▇▇▇█▆▇▆▆▇
model_3_bolshe_neyronov/valid_profit,▁▄▅▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇███████▇█▇▇█
model_3_bolshe_neyronov/valid_recall,████████▆▆▆▆▆▆▆▆▆▆▆▃▃▃▃▁▁▃▃▃▃▆
model_3_bolshe_neyronov/valid_roc_auc,▁▂▂▃▄▃▃▄▄▄▅▆▆▆▇▇▇▇▇██▇▇██▇▇▆▆█
model_3_bolshe_neyronov/train_loss,0.11184
model_3_bolshe_neyronov/valid_f05,0.62112
model_3_bolshe_neyronov/valid_precision,0.57143
model_3_bolshe_neyronov/valid_profit,-280
model_3_bolshe_neyronov/valid_recall,0.95238


In [66]:
train_metrics = evaluate_model(model_3, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_3, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9996
Precision: 0.8737
Recall: 1.0
F0.5-score: 0.8963
Profit: 115

Test
ROC-AUC: 0.9861
Precision: 0.6325
Recall: 0.8125
F0.5-score: 0.6618
Profit: -205930



Увеличение количества нейронов в 2 раза тоже улучшило качество. Однако profit все еще хуже бейзлайна

Из интересного, если увеличить колво нейронов в 4 раза, то качество станет хуже 

## model_4_tolko_batchnorm

In [67]:
EPOCHS = 30
SEED = 42

In [68]:
wandb.init(mode="disabled")
results = []
for LR in [0.05, 0.03, 0.01, 0.005, 0.002, 0.0001]:
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED) 

        model_4 = nn.Sequential(
            nn.Linear(9, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),

            nn.Linear(16, 8),
            nn.BatchNorm1d(8),
            nn.ReLU(),

            nn.Linear(8, 1)
        )

        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(model_4.parameters(), lr=LR)

        model_4 = train_model(
            model=model_4,
            train_loader=train_loader,
            X_valid=X_val,
            y_valid=y_val,
            loss_fn=loss_fn,
            optimizer=optimizer,
            epochs=EPOCHS,
            threshold=0.5,
            model_name="model_4_tolko_batchnorm"
        )

        val = evaluate_model(model_4, X_val, y_val, threshold=0.5, name="val")
        results.append({"lr": LR, "profit": val["profit"], "roc_auc": val["roc_auc"]})

model_4_tolko_batchnorm | valid epoch 1
ROC-AUC: 0.9754
Precision: 0.1429
Recall: 1.0
F0.5-score: 0.1724
Profit: -3045

Эпоха 1 | Train Loss: 0.9049 | Val ROC-AUC: 0.9754 | Val Profit: -3045
model_4_tolko_batchnorm | valid epoch 2
ROC-AUC: 0.9616
Precision: 0.2143
Recall: 1.0
F0.5-score: 0.2542
Profit: -1820

Эпоха 2 | Train Loss: 0.4806 | Val ROC-AUC: 0.9616 | Val Profit: -1820
model_4_tolko_batchnorm | valid epoch 3
ROC-AUC: 0.9646
Precision: 0.181
Recall: 1.0
F0.5-score: 0.2165
Profit: -2270

Эпоха 3 | Train Loss: 0.5201 | Val ROC-AUC: 0.9646 | Val Profit: -2270
model_4_tolko_batchnorm | valid epoch 4
ROC-AUC: 0.9667
Precision: 0.2308
Recall: 1.0
F0.5-score: 0.2727
Profit: -1645

Эпоха 4 | Train Loss: 0.4696 | Val ROC-AUC: 0.9667 | Val Profit: -1645
model_4_tolko_batchnorm | valid epoch 5
ROC-AUC: 0.9683
Precision: 0.5152
Recall: 0.8095
F0.5-score: 0.5556
Profit: -335

Эпоха 5 | Train Loss: 0.3298 | Val ROC-AUC: 0.9683 | Val Profit: -335
model_4_tolko_batchnorm | valid epoch 6
ROC-A

In [69]:
best = max(results, key=lambda r: r["profit"])
print("Лучшие гиперпараметры:", best)

Лучшие гиперпараметры: {'lr': 0.05, 'profit': np.int64(25), 'roc_auc': 0.9898054996646546}


In [70]:
LR = best["lr"]

In [71]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_4 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.ReLU(),

    nn.Linear(8, 1)
)

config = {
    "model": "MLP_batchnorm_only",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_4)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_4_batchnorm_only", config=config)
log_file = new_log_file()

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_4.parameters(), lr=LR)

model_4 = train_model(
    model=model_4,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_4_tolko_batchnorm"
)


save_results(model_4, "model_4", log_file, run)

run.finish()

model_4_tolko_batchnorm | valid epoch 1
ROC-AUC: 0.9754
Precision: 0.1429
Recall: 1.0
F0.5-score: 0.1724
Profit: -3045

Эпоха 1 | Train Loss: 0.9049 | Val ROC-AUC: 0.9754 | Val Profit: -3045
model_4_tolko_batchnorm | valid epoch 2
ROC-AUC: 0.9616
Precision: 0.2143
Recall: 1.0
F0.5-score: 0.2542
Profit: -1820

Эпоха 2 | Train Loss: 0.4806 | Val ROC-AUC: 0.9616 | Val Profit: -1820
model_4_tolko_batchnorm | valid epoch 3
ROC-AUC: 0.9646
Precision: 0.181
Recall: 1.0
F0.5-score: 0.2165
Profit: -2270

Эпоха 3 | Train Loss: 0.5201 | Val ROC-AUC: 0.9646 | Val Profit: -2270
model_4_tolko_batchnorm | valid epoch 4
ROC-AUC: 0.9667
Precision: 0.2308
Recall: 1.0
F0.5-score: 0.2727
Profit: -1645

Эпоха 4 | Train Loss: 0.4696 | Val ROC-AUC: 0.9667 | Val Profit: -1645
model_4_tolko_batchnorm | valid epoch 5
ROC-AUC: 0.9683
Precision: 0.5152
Recall: 0.8095
F0.5-score: 0.5556
Profit: -335

Эпоха 5 | Train Loss: 0.3298 | Val ROC-AUC: 0.9683 | Val Profit: -335
model_4_tolko_batchnorm | valid epoch 6
ROC-A

model_4_tolko_batchnorm/train_loss,█▄▄▄▃▃▂▂▃▂▂▂▃▂▁▂▁▁▂▂▂▁▁▁▂▂▁▁▁▁
model_4_tolko_batchnorm/valid_f05,▂▃▃▃▅▄▆▅▃▅▆▅▃▆▆▆▅▅▃▆▁▁▆▁▃▆▆█▆▆
model_4_tolko_batchnorm/valid_precision,▂▃▂▃▅▃▅▅▃▅▇▄▃▅▅▅▅▅▃▅▁▁▅▁▃▅▆█▅▅
model_4_tolko_batchnorm/valid_profit,▁▄▃▄▇▅▇▇▄▇█▇▅▇█▇▇▇▅▇██▇█▅▇███▇
model_4_tolko_batchnorm/valid_recall,████▇█▆▃█▆▃▆█▇▅▇▅▇██▁▁▆▁██▄▅▆▇
model_4_tolko_batchnorm/valid_roc_auc,▄▁▂▂▃▃▅▃▅▄▃▃▆▅▃▄▃▂▇▆▆▆▆█▇▇▇█▆▅
model_4_tolko_batchnorm/train_loss,0.20844
model_4_tolko_batchnorm/valid_f05,0.6391
model_4_tolko_batchnorm/valid_precision,0.60714
model_4_tolko_batchnorm/valid_profit,-210
model_4_tolko_batchnorm/valid_recall,0.80952


In [72]:
train_metrics = evaluate_model(model_4, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_4, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9938
Precision: 0.8958
Recall: 0.5181
F0.5-score: 0.7818
Profit: -110

Test
ROC-AUC: 0.9831
Precision: 0.8104
Recall: 0.4949
F0.5-score: 0.7187
Profit: -69905



Так как в предыдущие разы качество улучшилось с увеличением колво слоев, то теперь мы решили добавить колво слоев, а также после каждого слоя сделать только BatchNorm

Видно, что это сильно улучшило качество! И мы побили бейзлайн!

## model_5_tolko_dropout

In [86]:
EPOCHS = 40
SEED = 42

In [87]:
wandb.init(mode="disabled")
results = []
for LR in [0.01, 0.005, 0.001]:
    for DROPOUT_COEF in [0.05, 0.1, 0.015, 0.2, 0.025, 0.3]:
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED) 

        model_5 = nn.Sequential(
            nn.Linear(9, 128),
            nn.ReLU(),
            nn.Dropout(DROPOUT_COEF),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(DROPOUT_COEF),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(DROPOUT_COEF),

            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(DROPOUT_COEF),

            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Dropout(DROPOUT_COEF),

            nn.Linear(8, 1)
        )

        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(model_5.parameters(), lr=LR)
        model_5 = train_model(
            model=model_5,
            train_loader=train_loader,
            X_valid=X_val,
            y_valid=y_val,
            loss_fn=loss_fn,
            optimizer=optimizer,
            epochs=EPOCHS,
            threshold=0.5,
            model_name="model_5_tolko_dropout"
        )

        val = evaluate_model(model_5, X_val, y_val, threshold=0.5, name="val")
        results.append({"lr": LR, "dropout": DROPOUT_COEF, "profit": val["profit"], "roc_auc": val["roc_auc"]})


model_5_tolko_dropout | valid epoch 1
ROC-AUC: 0.9353
Precision: 0.2414
Recall: 1.0
F0.5-score: 0.2846
Profit: -1545

Эпоха 1 | Train Loss: 1.0889 | Val ROC-AUC: 0.9353 | Val Profit: -1545
model_5_tolko_dropout | valid epoch 2
ROC-AUC: 0.9204
Precision: 0.2121
Recall: 1.0
F0.5-score: 0.2518
Profit: -1845

Эпоха 2 | Train Loss: 0.8168 | Val ROC-AUC: 0.9204 | Val Profit: -1845
model_5_tolko_dropout | valid epoch 3
ROC-AUC: 0.9276
Precision: 0.2923
Recall: 0.9048
F0.5-score: 0.3381
Profit: -1065

Эпоха 3 | Train Loss: 0.6146 | Val ROC-AUC: 0.9276 | Val Profit: -1065
model_5_tolko_dropout | valid epoch 4
ROC-AUC: 0.9619
Precision: 0.3846
Recall: 0.9524
F0.5-score: 0.4367
Profit: -705

Эпоха 4 | Train Loss: 0.4525 | Val ROC-AUC: 0.9619 | Val Profit: -705
model_5_tolko_dropout | valid epoch 5
ROC-AUC: 0.9322
Precision: 0.274
Recall: 0.9524
F0.5-score: 0.3195
Profit: -1230

Эпоха 5 | Train Loss: 0.4916 | Val ROC-AUC: 0.9322 | Val Profit: -1230
model_5_tolko_dropout | valid epoch 6
ROC-AUC: 0.

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_5_tolko_dropout | valid epoch 7
ROC-AUC: 0.9649
Precision: 0.4043
Recall: 0.9048
F0.5-score: 0.4545
Profit: -615

Эпоха 7 | Train Loss: 0.3045 | Val ROC-AUC: 0.9649 | Val Profit: -615
model_5_tolko_dropout | valid epoch 8
ROC-AUC: 0.9634
Precision: 0.4
Recall: 0.9524
F0.5-score: 0.4525
Profit: -655

Эпоха 8 | Train Loss: 0.2563 | Val ROC-AUC: 0.9634 | Val Profit: -655
model_5_tolko_dropout | valid epoch 9
ROC-AUC: 0.9619
Precision: 0.3443
Recall: 1.0
F0.5-score: 0.3962
Profit: -895

Эпоха 9 | Train Loss: 0.2621 | Val ROC-AUC: 0.9619 | Val Profit: -895
model_5_tolko_dropout | valid epoch 10
ROC-AUC: 0.9686
Precision: 0.3922
Recall: 0.9524
F0.5-score: 0.4444
Profit: -680

Эпоха 10 | Train Loss: 0.2376 | Val ROC-AUC: 0.9686 | Val Profit: -680
model_5_tolko_dropout | valid epoch 11
ROC-AUC: 0.9596
Precision: 0.2923
Recall: 0.9048
F0.5-score: 0.3381
Profit: -1065

Эпоха 11 | Train Loss: 0.2943 | Val ROC-AUC: 0.9596 | Val Profit: -1065
model_5_tolko_dropout | valid epoch 12
ROC-AUC: 0.

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_5_tolko_dropout | valid epoch 8
ROC-AUC: 0.9557
Precision: 0.3725
Recall: 0.9048
F0.5-score: 0.4222
Profit: -715

Эпоха 8 | Train Loss: 0.3170 | Val ROC-AUC: 0.9557 | Val Profit: -715
model_5_tolko_dropout | valid epoch 9
ROC-AUC: 0.9612
Precision: 0.3455
Recall: 0.9048
F0.5-score: 0.3942
Profit: -815

Эпоха 9 | Train Loss: 0.3091 | Val ROC-AUC: 0.9612 | Val Profit: -815
model_5_tolko_dropout | valid epoch 10
ROC-AUC: 0.9584
Precision: 0.4286
Recall: 0.8571
F0.5-score: 0.4762
Profit: -525

Эпоха 10 | Train Loss: 0.3011 | Val ROC-AUC: 0.9584 | Val Profit: -525
model_5_tolko_dropout | valid epoch 11
ROC-AUC: 0.959
Precision: 0.4
Recall: 0.8571
F0.5-score: 0.4478
Profit: -600

Эпоха 11 | Train Loss: 0.2782 | Val ROC-AUC: 0.9590 | Val Profit: -600
model_5_tolko_dropout | valid epoch 12
ROC-AUC: 0.9623
Precision: 0.339
Recall: 0.9524
F0.5-score: 0.3891
Profit: -880

Эпоха 12 | Train Loss: 0.2890 | Val ROC-AUC: 0.9623 | Val Profit: -880
model_5_tolko_dropout | valid epoch 13
ROC-AUC: 0

...

model_5_tolko_dropout | valid epoch 5
ROC-AUC: 0.9508
Precision: 0.2838
Recall: 1.0
F0.5-score: 0.3312
Profit: -1220

Эпоха 5 | Train Loss: 0.6292 | Val ROC-AUC: 0.9508 | Val Profit: -1220
model_5_tolko_dropout | valid epoch 6
ROC-AUC: 0.9522
Precision: 0.2917
Recall: 1.0
F0.5-score: 0.3398
Profit: -1170

Эпоха 6 | Train Loss: 0.4478 | Val ROC-AUC: 0.9522 | Val Profit: -1170
model_5_tolko_dropout | valid epoch 7
ROC-AUC: 0.9608
Precision: 0.3333
Recall: 0.9048
F0.5-score: 0.3815
Profit: -865

Эпоха 7 | Train Loss: 0.3738 | Val ROC-AUC: 0.9608 | Val Profit: -865
model_5_tolko_dropout | valid epoch 8
ROC-AUC: 0.9586
Precision: 0.3878
Recall: 0.9048
F0.5-score: 0.4378
Profit: -665

Эпоха 8 | Train Loss: 0.3796 | Val ROC-AUC: 0.9586 | Val Profit: -665
model_5_tolko_dropout | valid epoch 9
ROC-AUC: 0.9568
Precision: 0.4118
Recall: 0.6667
F0.5-score: 0.4459
Profit: -465

Эпоха 9 | Train Loss: 0.3441 | Val ROC-AUC: 0.9568 | Val Profit: -465
model_5_tolko_dropout | valid epoch 10
ROC-AUC: 0.96

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_5_tolko_dropout | valid epoch 2
ROC-AUC: 0.947
Precision: 0.3902
Recall: 0.7619
F0.5-score: 0.4324
Profit: -570

Эпоха 2 | Train Loss: 1.2556 | Val ROC-AUC: 0.9470 | Val Profit: -570
model_5_tolko_dropout | valid epoch 3
ROC-AUC: 0.9521
Precision: 0.2899
Recall: 0.9524
F0.5-score: 0.3367
Profit: -1130

Эпоха 3 | Train Loss: 1.0387 | Val ROC-AUC: 0.9521 | Val Profit: -1130
model_5_tolko_dropout | valid epoch 4
ROC-AUC: 0.9317
Precision: 0.253
Recall: 1.0
F0.5-score: 0.2975
Profit: -1445

Эпоха 4 | Train Loss: 0.8100 | Val ROC-AUC: 0.9317 | Val Profit: -1445
model_5_tolko_dropout | valid epoch 5
ROC-AUC: 0.9423
Precision: 0.274
Recall: 0.9524
F0.5-score: 0.3195
Profit: -1230

Эпоха 5 | Train Loss: 0.6128 | Val ROC-AUC: 0.9423 | Val Profit: -1230
model_5_tolko_dropout | valid epoch 6
ROC-AUC: 0.9517
Precision: 0.2879
Recall: 0.9048
F0.5-score: 0.3333
Profit: -1090

Эпоха 6 | Train Loss: 0.4255 | Val ROC-AUC: 0.9517 | Val Profit: -1090
model_5_tolko_dropout | valid epoch 7
ROC-AUC: 0

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_5_tolko_dropout | valid epoch 7
ROC-AUC: 0.963
Precision: 0.3519
Recall: 0.9048
F0.5-score: 0.4008
Profit: -790

Эпоха 7 | Train Loss: 0.4533 | Val ROC-AUC: 0.9630 | Val Profit: -790
model_5_tolko_dropout | valid epoch 8
ROC-AUC: 0.9604
Precision: 0.3387
Recall: 1.0
F0.5-score: 0.3903
Profit: -920

Эпоха 8 | Train Loss: 0.4002 | Val ROC-AUC: 0.9604 | Val Profit: -920
model_5_tolko_dropout | valid epoch 9
ROC-AUC: 0.961
Precision: 0.413
Recall: 0.9048
F0.5-score: 0.4634
Profit: -590

Эпоха 9 | Train Loss: 0.3737 | Val ROC-AUC: 0.9610 | Val Profit: -590
model_5_tolko_dropout | valid epoch 10
ROC-AUC: 0.9615
Precision: 0.4815
Recall: 0.619
F0.5-score: 0.5039
Profit: -325

Эпоха 10 | Train Loss: 0.3062 | Val ROC-AUC: 0.9615 | Val Profit: -325
model_5_tolko_dropout | valid epoch 11
ROC-AUC: 0.9604
Precision: 0.4
Recall: 0.6667
F0.5-score: 0.4348
Profit: -490

Эпоха 11 | Train Loss: 0.2362 | Val ROC-AUC: 0.9604 | Val Profit: -490
model_5_tolko_dropout | valid epoch 12
ROC-AUC: 0.9622
P

In [88]:
best = max(results, key=lambda r: r["profit"])
print("Лучшие гиперпараметры:", best)

Лучшие гиперпараметры: {'lr': 0.005, 'dropout': 0.015, 'profit': np.int64(-80), 'roc_auc': 0.9856472166331322}


In [89]:
LR = best["lr"]
DROPOUT_COEF = best["dropout"]

In [90]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_5 = nn.Sequential(
    nn.Linear(9, 128),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_5.parameters(), lr=LR)

config = {
    "model": "MLP_dropout_only",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "dropout_coef": DROPOUT_COEF,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_5)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_5_dropout_only", config=config)
log_file = new_log_file()

model_5 = train_model(
    model=model_5,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_5_tolko_dropout"
)

save_results(model_5, "model_5", log_file, run)

run.finish()


model_5_tolko_dropout | valid epoch 1
ROC-AUC: 0.9375
Precision: 0.3077
Recall: 0.9524
F0.5-score: 0.3559
Profit: -1030

Эпоха 1 | Train Loss: 1.1719 | Val ROC-AUC: 0.9375 | Val Profit: -1030
model_5_tolko_dropout | valid epoch 2
ROC-AUC: 0.9333
Precision: 0.1909
Recall: 1.0
F0.5-score: 0.2278
Profit: -2120

Эпоха 2 | Train Loss: 0.7785 | Val ROC-AUC: 0.9333 | Val Profit: -2120
model_5_tolko_dropout | valid epoch 3
ROC-AUC: 0.9586
Precision: 0.2471
Recall: 1.0
F0.5-score: 0.2909
Profit: -1495

Эпоха 3 | Train Loss: 0.5299 | Val ROC-AUC: 0.9586 | Val Profit: -1495
model_5_tolko_dropout | valid epoch 4
ROC-AUC: 0.9518
Precision: 0.2039
Recall: 1.0
F0.5-score: 0.2425
Profit: -1945

Эпоха 4 | Train Loss: 0.5074 | Val ROC-AUC: 0.9518 | Val Profit: -1945
model_5_tolko_dropout | valid epoch 5
ROC-AUC: 0.9549
Precision: 0.2333
Recall: 1.0
F0.5-score: 0.2756
Profit: -1620

Эпоха 5 | Train Loss: 0.4923 | Val ROC-AUC: 0.9549 | Val Profit: -1620
model_5_tolko_dropout | valid epoch 6
ROC-AUC: 0.952

model_5_tolko_dropout/train_loss,█▅▄▄▄▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▃▂▂▂▂▂▁▂▂▂▁▁▁▂▁▁▁▁▁
model_5_tolko_dropout/valid_f05,▃▁▂▁▂▂▂▂▄▃▃▄▄▃▃▃▃▃▃▄▂▂▄▅▃▂▃▅▂▂▄▅▅▄▄▅▅▆▇█
model_5_tolko_dropout/valid_precision,▂▁▂▁▂▂▂▂▃▃▃▄▄▃▂▃▂▂▃▄▂▂▄▄▃▂▃▅▂▂▃▄▄▄▄▄▅▆▇█
model_5_tolko_dropout/valid_profit,▅▁▃▂▃▄▄▅▆▆▅▆▆▅▅▅▅▄▅▆▅▄▆▇▅▄▆▇▄▄▆▇▇▆▆▆▇▇██
model_5_tolko_dropout/valid_recall,▇████▆█▆▆▇▆▆▅███▇█▇▆▆█▆▆██▆▃█▇▆▆▇▆▆▇▅▂▂▁
model_5_tolko_dropout/valid_roc_auc,▂▁▄▃▄▄▅▄▆▅▅▄▅▅▅▆▅▅▅▅▃▆▇▆▆▅▆▇▅▄▆▇▇▅▅▇▇▇██
model_5_tolko_dropout/train_loss,0.09129
model_5_tolko_dropout/valid_f05,0.74257
model_5_tolko_dropout/valid_precision,0.75
model_5_tolko_dropout/valid_profit,-80
model_5_tolko_dropout/valid_recall,0.71429


In [91]:
train_metrics = evaluate_model(model_5, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_5, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9959
Precision: 0.8642
Recall: 0.8434
F0.5-score: 0.86
Profit: 10

Test
ROC-AUC: 0.9804
Precision: 0.7128
Recall: 0.6384
F0.5-score: 0.6966
Profit: -119740



В этом эксперименте добавили только dropout и подбирали его коэффициент перебором. Лучшее качество получилось при dropout = 0.015

Увеличение колво эпох не улучшило ситуацию с переобучением

Это чуть хуже бейзлайна, но батчнорм показал себя лучше, чем дропаут в этом эксперименте

## model_6_batchnorm_i_dropout

In [92]:
EPOCHS = 30
SEED = 42

In [93]:
wandb.init(mode="disabled")
results = []
for LR in [0.01, 0.005, 0.001]:
    for DROPOUT_COEF in [0.05, 0.1, 0.015, 0.2, 0.025, 0.3]:
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED) 

        model_6 = nn.Sequential(
            nn.Linear(9, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(DROPOUT_COEF),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(DROPOUT_COEF),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(DROPOUT_COEF),

            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(DROPOUT_COEF),

            nn.Linear(16, 8),
            nn.BatchNorm1d(8),
            nn.ReLU(),
            nn.Dropout(DROPOUT_COEF),

            nn.Linear(8, 1)
        )

        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(model_6.parameters(), lr=LR)
        model_6 = train_model(
            model=model_6,
            train_loader=train_loader,
            X_valid=X_val,
            y_valid=y_val,
            loss_fn=loss_fn,
            optimizer=optimizer,
            epochs=EPOCHS,
            threshold=0.5,
            model_name="model_6_batchnorm_i_dropout"
        )

        val = evaluate_model(model_6, X_val, y_val, threshold=0.5, name="val")
        results.append({"lr": LR, "dropout": DROPOUT_COEF, "profit": val["profit"], "roc_auc": val["roc_auc"]})


model_6_batchnorm_i_dropout | valid epoch 1
ROC-AUC: 0.9635
Precision: 0.1346
Recall: 1.0
F0.5-score: 0.1628
Profit: -3270

Эпоха 1 | Train Loss: 1.0119 | Val ROC-AUC: 0.9635 | Val Profit: -3270
model_6_batchnorm_i_dropout | valid epoch 2
ROC-AUC: 0.9581
Precision: 0.2308
Recall: 1.0
F0.5-score: 0.2727
Profit: -1645

Эпоха 2 | Train Loss: 0.5804 | Val ROC-AUC: 0.9581 | Val Profit: -1645
model_6_batchnorm_i_dropout | valid epoch 3
ROC-AUC: 0.9683
Precision: 0.2763
Recall: 1.0
F0.5-score: 0.3231
Profit: -1270

Эпоха 3 | Train Loss: 0.3780 | Val ROC-AUC: 0.9683 | Val Profit: -1270
model_6_batchnorm_i_dropout | valid epoch 4
ROC-AUC: 0.9602
Precision: 0.3231
Recall: 1.0
F0.5-score: 0.3737
Profit: -995

Эпоха 4 | Train Loss: 0.3449 | Val ROC-AUC: 0.9602 | Val Profit: -995
model_6_batchnorm_i_dropout | valid epoch 5
ROC-AUC: 0.9347
Precision: 0.2647
Recall: 0.8571
F0.5-score: 0.3072
Profit: -1175

Эпоха 5 | Train Loss: 0.3749 | Val ROC-AUC: 0.9347 | Val Profit: -1175
model_6_batchnorm_i_drop

In [94]:
best = max(results, key=lambda r: r["profit"])
print("Лучшие гиперпараметры:", best)

Лучшие гиперпараметры: {'lr': 0.005, 'dropout': 0.015, 'profit': np.int64(-30), 'roc_auc': 0.9830985915492958}


In [95]:
LR = best["lr"]
DROPOUT_COEF = best["dropout"]

In [96]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_6 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_6.parameters(), lr=LR)

config = {
    "model": "MLP_batchnorm_dropout",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "dropout_coef": DROPOUT_COEF,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_6)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_6_batchnorm_dropout", config=config)
log_file = new_log_file()

model_6 = train_model(
    model=model_6,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_6_batchnorm_i_dropout"
)


save_results(model_6, "model_6", log_file, run)

run.finish()

model_6_batchnorm_i_dropout | valid epoch 1
ROC-AUC: 0.9485
Precision: 0.1469
Recall: 1.0
F0.5-score: 0.1771
Profit: -2945

Эпоха 1 | Train Loss: 1.1451 | Val ROC-AUC: 0.9485 | Val Profit: -2945
model_6_batchnorm_i_dropout | valid epoch 2
ROC-AUC: 0.9693
Precision: 0.1927
Recall: 1.0
F0.5-score: 0.2298
Profit: -2095

Эпоха 2 | Train Loss: 0.7664 | Val ROC-AUC: 0.9693 | Val Profit: -2095
model_6_batchnorm_i_dropout | valid epoch 3
ROC-AUC: 0.9646
Precision: 0.3043
Recall: 1.0
F0.5-score: 0.3535
Profit: -1095

Эпоха 3 | Train Loss: 0.5423 | Val ROC-AUC: 0.9646 | Val Profit: -1095
model_6_batchnorm_i_dropout | valid epoch 4
ROC-AUC: 0.9732
Precision: 0.4
Recall: 0.9524
F0.5-score: 0.4525
Profit: -655

Эпоха 4 | Train Loss: 0.3972 | Val ROC-AUC: 0.9732 | Val Profit: -655
model_6_batchnorm_i_dropout | valid epoch 5
ROC-AUC: 0.9626
Precision: 0.4348
Recall: 0.9524
F0.5-score: 0.4878
Profit: -555

Эпоха 5 | Train Loss: 0.2974 | Val ROC-AUC: 0.9626 | Val Profit: -555
model_6_batchnorm_i_dropou

model_6_batchnorm_i_dropout/train_loss,█▆▄▃▃▃▃▃▂▂▂▃▃▂▂▁▂▃▂▂▂▁▁▁▂▁▁▁▁▁
model_6_batchnorm_i_dropout/valid_f05,▁▂▃▄▄▃▂▄▅▆▆▃▄▅▆▇▄▃▅▆▇▇██▅▇▆▇▆▂
model_6_batchnorm_i_dropout/valid_precision,▁▁▃▄▄▃▂▃▅▆▆▃▃▅▆▇▄▃▄▅▇▇██▄▆▇▇▆▄
model_6_batchnorm_i_dropout/valid_profit,▁▃▅▆▇▅▄▆▇██▆▆▇██▇▆▇█████▇█████
model_6_batchnorm_i_dropout/valid_recall,████████▆▆▅██▇▆▅▇▇▇▆▆▅▅▆▆▆▄▄▄▁
model_6_batchnorm_i_dropout/valid_roc_auc,▅▇▇▇▆▇▇▇▇▇▇▇▇▇█▆▇▇█▇▇▇██▆█▇█▇▁
model_6_batchnorm_i_dropout/train_loss,0.0554
model_6_batchnorm_i_dropout/valid_f05,0.30612
model_6_batchnorm_i_dropout/valid_precision,0.42857
model_6_batchnorm_i_dropout/valid_profit,-175
model_6_batchnorm_i_dropout/valid_recall,0.14286


In [97]:
train_metrics = evaluate_model(model_6, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_6, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9948
Precision: 0.8852
Recall: 0.6506
F0.5-score: 0.8257
Profit: -50

Test
ROC-AUC: 0.9803
Precision: 0.778
Recall: 0.5243
F0.5-score: 0.7094
Profit: -82975



Если соединить 2 и дропаут и батчнорм вместе, то качество получается хорошим. Но оно хуже, чем только при использовании batchnorm

## model_7_leaky_relu

In [104]:
EPOCHS = 30
SEED = 42

In [105]:
wandb.init(mode="disabled")
results = []
for LR in [0.01, 0.005, 0.001]:
    for DROPOUT_COEF in [0, 0.05, 0.01, 0.015, 0.2, 0.025, 0.3]:
        for NEG_SLOPE in [0.01, 0.03, 0.05, 0.1]:
            random.seed(SEED)
            np.random.seed(SEED)
            torch.manual_seed(SEED) 

            model_7 = nn.Sequential(
                nn.Linear(9, 128),
                nn.BatchNorm1d(128),
                nn.LeakyReLU(negative_slope=NEG_SLOPE),
                nn.Dropout(DROPOUT_COEF),

                nn.Linear(128, 64),
                nn.BatchNorm1d(64),
                nn.LeakyReLU(negative_slope=NEG_SLOPE),
                nn.Dropout(DROPOUT_COEF),

                nn.Linear(64, 32),
                nn.BatchNorm1d(32),
                nn.LeakyReLU(negative_slope=NEG_SLOPE),
                nn.Dropout(DROPOUT_COEF),

                nn.Linear(32, 16),
                nn.BatchNorm1d(16),
                nn.LeakyReLU(negative_slope=NEG_SLOPE),
                nn.Dropout(DROPOUT_COEF),

                nn.Linear(16, 8),
                nn.BatchNorm1d(8),
                nn.LeakyReLU(negative_slope=NEG_SLOPE),
                nn.Dropout(DROPOUT_COEF),

                nn.Linear(8, 1)
            )

            loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            optimizer = torch.optim.Adam(model_7.parameters(), lr=LR)
            model_7 = train_model(
                model=model_7,
                train_loader=train_loader,
                X_valid=X_val,
                y_valid=y_val,
                loss_fn=loss_fn,
                optimizer=optimizer,
                epochs=EPOCHS,
                threshold=0.5,
                model_name="model_7_leaky_relu"
            )

            val = evaluate_model(model_7, X_val, y_val, threshold=0.5, name="val")
            results.append({"lr": LR, "dropout": DROPOUT_COEF, "neg_slope": NEG_SLOPE, "profit": val["profit"], "roc_auc": val["roc_auc"]})

model_7_leaky_relu | valid epoch 1
ROC-AUC: 0.9438
Precision: 0.1364
Recall: 1.0
F0.5-score: 0.1648
Profit: -3220

Эпоха 1 | Train Loss: 1.0457 | Val ROC-AUC: 0.9438 | Val Profit: -3220
model_7_leaky_relu | valid epoch 2
ROC-AUC: 0.9632
Precision: 0.2165
Recall: 1.0
F0.5-score: 0.2567
Profit: -1795

Эпоха 2 | Train Loss: 0.5750 | Val ROC-AUC: 0.9632 | Val Profit: -1795
model_7_leaky_relu | valid epoch 3
ROC-AUC: 0.9537
Precision: 0.2958
Recall: 1.0
F0.5-score: 0.3443
Profit: -1145

Эпоха 3 | Train Loss: 0.3804 | Val ROC-AUC: 0.9537 | Val Profit: -1145
model_7_leaky_relu | valid epoch 4
ROC-AUC: 0.9576
Precision: 0.241
Recall: 0.9524
F0.5-score: 0.2833
Profit: -1480

Эпоха 4 | Train Loss: 0.3889 | Val ROC-AUC: 0.9576 | Val Profit: -1480
model_7_leaky_relu | valid epoch 5
ROC-AUC: 0.9757
Precision: 0.475
Recall: 0.9048
F0.5-score: 0.5249
Profit: -440

Эпоха 5 | Train Loss: 0.3124 | Val ROC-AUC: 0.9757 | Val Profit: -440
model_7_leaky_relu | valid epoch 6
ROC-AUC: 0.9733
Precision: 0.3226

In [106]:
best = max(results, key=lambda r: r["profit"])
print("Лучшие гиперпараметры:", best)

Лучшие гиперпараметры: {'lr': 0.01, 'dropout': 0.015, 'neg_slope': 0.1, 'profit': np.int64(-55), 'roc_auc': 0.9849765258215962}


In [107]:
LR = best["lr"]
DROPOUT_COEF = best["dropout"]
NEG_SLOPE = best["neg_slope"]

In [108]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_7 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.LeakyReLU(negative_slope=NEG_SLOPE),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.LeakyReLU(negative_slope=NEG_SLOPE),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.LeakyReLU(negative_slope=NEG_SLOPE),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.LeakyReLU(negative_slope=NEG_SLOPE),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.LeakyReLU(negative_slope=NEG_SLOPE),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_7.parameters(), lr=LR)

config = {
    "model": "MLP_leaky_relu",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "dropout_coef": DROPOUT_COEF,
    "negative_slope": NEG_SLOPE,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_7)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_7_leaky_relu", config=config)
log_file = new_log_file()

model_7 = train_model(
    model=model_7,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_7_leaky_relu"
)
save_results(model_7, "model_7", log_file, run)

run.finish()

model_7_leaky_relu | valid epoch 1
ROC-AUC: 0.9434
Precision: 0.1373
Recall: 1.0
F0.5-score: 0.1659
Profit: -3195

Эпоха 1 | Train Loss: 1.0064 | Val ROC-AUC: 0.9434 | Val Profit: -3195
model_7_leaky_relu | valid epoch 2
ROC-AUC: 0.9572
Precision: 0.1875
Recall: 1.0
F0.5-score: 0.2239
Profit: -2170

Эпоха 2 | Train Loss: 0.5595 | Val ROC-AUC: 0.9572 | Val Profit: -2170
model_7_leaky_relu | valid epoch 3
ROC-AUC: 0.9509
Precision: 0.2658
Recall: 1.0
F0.5-score: 0.3116
Profit: -1345

Эпоха 3 | Train Loss: 0.4192 | Val ROC-AUC: 0.9509 | Val Profit: -1345
model_7_leaky_relu | valid epoch 4
ROC-AUC: 0.965
Precision: 0.2121
Recall: 1.0
F0.5-score: 0.2518
Profit: -1845

Эпоха 4 | Train Loss: 0.4338 | Val ROC-AUC: 0.9650 | Val Profit: -1845
model_7_leaky_relu | valid epoch 5
ROC-AUC: 0.9734
Precision: 0.3704
Recall: 0.9524
F0.5-score: 0.4219
Profit: -755

Эпоха 5 | Train Loss: 0.3231 | Val ROC-AUC: 0.9734 | Val Profit: -755
model_7_leaky_relu | valid epoch 6
ROC-AUC: 0.9675
Precision: 0.4375
R

model_7_leaky_relu/train_loss,█▅▄▄▃▂▃▂▃▄▂▂▂▂▁▁▂▂▁▁▂▃▂▂▂▁▁▁▁▁
model_7_leaky_relu/valid_f05,▁▂▃▂▄▅▄▆▃▃▅▆█▇█▆▆▇▄▅▄▃▅▇▇▇▃▃▄▇
model_7_leaky_relu/valid_precision,▁▂▂▂▃▄▃▅▂▂▄▅▇▇█▅▅▇▆▆▃▃▄▆▆▆▄▄▆▆
model_7_leaky_relu/valid_profit,▁▃▅▄▆▇▆▇▅▅▇▇███▇▇███▆▆▇███████
model_7_leaky_relu/valid_recall,█████▅█████▆▅▄▄▇█▅▁▂███▅▅▄▁▁▁▇
model_7_leaky_relu/valid_roc_auc,▃▅▄▆▇▆▆▇▁▆▆▆▇▇█▇████▆▅▇▇▆▆▇▇██
model_7_leaky_relu/train_loss,0.08178
model_7_leaky_relu/valid_f05,0.67376
model_7_leaky_relu/valid_precision,0.63333
model_7_leaky_relu/valid_profit,-190
model_7_leaky_relu/valid_recall,0.90476


In [109]:
train_metrics = evaluate_model(model_7, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_7, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9943
Precision: 0.9
Recall: 0.4337
F0.5-score: 0.7407
Profit: -155

Test
ROC-AUC: 0.9785
Precision: 0.8168
Recall: 0.3803
F0.5-score: 0.6643
Profit: -79005



Поменяли функцию активации с ReLU на LeakyReLU и перебрали LR, DROPOUT_COEF и NEG_SLOPE. Получилось добиться лучшего качества по Profit в -79к

Да, это лучше бейзлайна, но качество ухудшилось. 

Вывод: особого эффекта LeakyRelu не дал

## model_8_weight_decay

In [137]:
EPOCHS = 40
SEED = 42

In [138]:
wandb.init(mode="disabled")
results = []
for LR in [0.01, 0.005, 0.001]:
    for DROPOUT_COEF in [0, 0.05, 0.01, 0.015, 0.2, 0.025, 0.3]:
        for WEIGHT_DECAY in [0.0001, 0.001, 0.01]:
            random.seed(SEED)
            np.random.seed(SEED)
            torch.manual_seed(SEED) 

            model_8 = nn.Sequential(
                nn.Linear(9, 128),
                nn.BatchNorm1d(128),
                nn.ReLU(),
                nn.Dropout(DROPOUT_COEF),

                nn.Linear(128, 64),
                nn.BatchNorm1d(64),
                nn.ReLU(),
                nn.Dropout(DROPOUT_COEF),

                nn.Linear(64, 32),
                nn.BatchNorm1d(32),
                nn.ReLU(),
                nn.Dropout(DROPOUT_COEF),

                nn.Linear(32, 16),
                nn.BatchNorm1d(16),
                nn.ReLU(),
                nn.Dropout(DROPOUT_COEF),

                nn.Linear(16, 8),
                nn.BatchNorm1d(8),
                nn.ReLU(),
                nn.Dropout(DROPOUT_COEF),

                nn.Linear(8, 1)
            )

            loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            optimizer = torch.optim.Adam(model_8.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
            model_8 = train_model(
                model=model_8,
                train_loader=train_loader,
                X_valid=X_val,
                y_valid=y_val,
                loss_fn=loss_fn,
                optimizer=optimizer,
                epochs=EPOCHS,
                threshold=0.5,
                model_name="model_8_weight_decay"
            )

            val = evaluate_model(model_8, X_val, y_val, threshold=0.5, name="val")
            results.append({"lr": LR, "dropout": DROPOUT_COEF, "weight_decay": WEIGHT_DECAY, "profit": val["profit"], "roc_auc": val["roc_auc"]})

model_8_weight_decay | valid epoch 1
ROC-AUC: 0.9522
Precision: 0.1235
Recall: 1.0
F0.5-score: 0.1498
Profit: -3620

Эпоха 1 | Train Loss: 1.0011 | Val ROC-AUC: 0.9522 | Val Profit: -3620
model_8_weight_decay | valid epoch 2
ROC-AUC: 0.9606
Precision: 0.2165
Recall: 1.0
F0.5-score: 0.2567
Profit: -1795

Эпоха 2 | Train Loss: 0.5483 | Val ROC-AUC: 0.9606 | Val Profit: -1795
model_8_weight_decay | valid epoch 3
ROC-AUC: 0.9645
Precision: 0.2386
Recall: 1.0
F0.5-score: 0.2815
Profit: -1570

Эпоха 3 | Train Loss: 0.4441 | Val ROC-AUC: 0.9645 | Val Profit: -1570
model_8_weight_decay | valid epoch 4
ROC-AUC: 0.9634
Precision: 0.3585
Recall: 0.9048
F0.5-score: 0.4077
Profit: -765

Эпоха 4 | Train Loss: 0.3131 | Val ROC-AUC: 0.9634 | Val Profit: -765
model_8_weight_decay | valid epoch 5
ROC-AUC: 0.9638
Precision: 0.3621
Recall: 1.0
F0.5-score: 0.415
Profit: -820

Эпоха 5 | Train Loss: 0.3646 | Val ROC-AUC: 0.9638 | Val Profit: -820
model_8_weight_decay | valid epoch 6
ROC-AUC: 0.9718
Precision

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_8_weight_decay | valid epoch 9
ROC-AUC: 0.9795
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 9 | Train Loss: 0.2918 | Val ROC-AUC: 0.9795 | Val Profit: -105
model_8_weight_decay | valid epoch 10
ROC-AUC: 0.9805
Precision: 0.6071
Recall: 0.8095
F0.5-score: 0.6391
Profit: -210

Эпоха 10 | Train Loss: 0.2954 | Val ROC-AUC: 0.9805 | Val Profit: -210
model_8_weight_decay | valid epoch 11
ROC-AUC: 0.9624
Precision: 0.3333
Recall: 0.0952
F0.5-score: 0.2222
Profit: -185

Эпоха 11 | Train Loss: 0.2974 | Val ROC-AUC: 0.9624 | Val Profit: -185
model_8_weight_decay | valid epoch 12
ROC-AUC: 0.9681
Precision: 0.3182
Recall: 1.0
F0.5-score: 0.3684
Profit: -1020

Эпоха 12 | Train Loss: 0.3128 | Val ROC-AUC: 0.9681 | Val Profit: -1020
model_8_weight_decay | valid epoch 13
ROC-AUC: 0.9791
Precision: 0.6667
Recall: 0.6667
F0.5-score: 0.6667
Profit: -140

Эпоха 13 | Train Loss: 0.3064 | Val ROC-AUC: 0.9791 | Val Profit: -140


/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_8_weight_decay | valid epoch 14
ROC-AUC: 0.9764
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 14 | Train Loss: 0.2854 | Val ROC-AUC: 0.9764 | Val Profit: -105
model_8_weight_decay | valid epoch 15
ROC-AUC: 0.9855
Precision: 0.75
Recall: 0.2857
F0.5-score: 0.566
Profit: -95

Эпоха 15 | Train Loss: 0.2496 | Val ROC-AUC: 0.9855 | Val Profit: -95
model_8_weight_decay | valid epoch 16
ROC-AUC: 0.9708
Precision: 0.2414
Recall: 1.0
F0.5-score: 0.2846
Profit: -1545

Эпоха 16 | Train Loss: 0.3036 | Val ROC-AUC: 0.9708 | Val Profit: -1545
model_8_weight_decay | valid epoch 17
ROC-AUC: 0.9765
Precision: 0.619
Recall: 0.619
F0.5-score: 0.619
Profit: -175

Эпоха 17 | Train Loss: 0.2606 | Val ROC-AUC: 0.9765 | Val Profit: -175
model_8_weight_decay | valid epoch 18
ROC-AUC: 0.9768
Precision: 0.5517
Recall: 0.7619
F0.5-score: 0.5839
Profit: -270

Эпоха 18 | Train Loss: 0.2449 | Val ROC-AUC: 0.9768 | Val Profit: -270
model_8_weight_decay | valid epoch 19
ROC-AUC: 0.9784
Precision

...

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_8_weight_decay | valid epoch 26
ROC-AUC: 0.8879
Precision: 0.5
Recall: 0.0952
F0.5-score: 0.2703
Profit: -135

Эпоха 26 | Train Loss: 0.1518 | Val ROC-AUC: 0.8879 | Val Profit: -135
model_8_weight_decay | valid epoch 27
ROC-AUC: 0.8317
Precision: 1.0
Recall: 0.0476
F0.5-score: 0.2
Profit: -95

Эпоха 27 | Train Loss: 0.1402 | Val ROC-AUC: 0.8317 | Val Profit: -95
model_8_weight_decay | valid epoch 28
ROC-AUC: 0.9093
Precision: 0.3333
Recall: 0.0476
F0.5-score: 0.1515
Profit: -145

Эпоха 28 | Train Loss: 0.1217 | Val ROC-AUC: 0.9093 | Val Profit: -145
model_8_weight_decay | valid epoch 29
ROC-AUC: 0.8712
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -155

Эпоха 29 | Train Loss: 0.1274 | Val ROC-AUC: 0.8712 | Val Profit: -155
model_8_weight_decay | valid epoch 30
ROC-AUC: 0.8322
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 30 | Train Loss: 0.1018 | Val ROC-AUC: 0.8322 | Val Profit: -105


/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

model_8_weight_decay | valid epoch 31
ROC-AUC: 0.861
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 31 | Train Loss: 0.0906 | Val ROC-AUC: 0.8610 | Val Profit: -105
model_8_weight_decay | valid epoch 32
ROC-AUC: 0.8598
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 32 | Train Loss: 0.0866 | Val ROC-AUC: 0.8598 | Val Profit: -105
model_8_weight_decay | valid epoch 33
ROC-AUC: 0.9092
Precision: 0.3333
Recall: 0.0476
F0.5-score: 0.1515
Profit: -145

Эпоха 33 | Train Loss: 0.0786 | Val ROC-AUC: 0.9092 | Val Profit: -145
model_8_weight_decay | valid epoch 34
ROC-AUC: 0.8177
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 34 | Train Loss: 0.0778 | Val ROC-AUC: 0.8177 | Val Profit: -105
model_8_weight_decay | valid epoch 35
ROC-AUC: 0.8463
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -130

Эпоха 35 | Train Loss: 0.0764 | Val ROC-AUC: 0.8463 | Val Profit: -130
model_8_weight_decay | valid epoch 36
ROC-AUC: 0.877
Precision: 0.3333
Recall: 0.0

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [139]:
best = max(results, key=lambda r: r["profit"])
print("Лучшие гиперпараметры:", best)

Лучшие гиперпараметры: {'lr': 0.005, 'dropout': 0.01, 'weight_decay': 0.01, 'profit': np.int64(-40), 'roc_auc': 0.9849765258215962}


In [140]:
LR = best["lr"]
WEIGHT_DECAY = best["weight_decay"]
DROPOUT_COEF = best["dropout"]

In [141]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_8 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model_8.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

config = {
    "model": "MLP_weight_decay",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_8)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_8_weight_decay", config=config)
log_file = new_log_file()

model_8 = train_model(
    model=model_8,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_8_weight_decay"
)

save_results(model_8, "model_8", log_file, run)

run.finish()

model_8_weight_decay | valid epoch 1
ROC-AUC: 0.9437
Precision: 0.1765
Recall: 1.0
F0.5-score: 0.2113
Profit: -2345

Эпоха 1 | Train Loss: 1.1563 | Val ROC-AUC: 0.9437 | Val Profit: -2345
model_8_weight_decay | valid epoch 2
ROC-AUC: 0.9596
Precision: 0.2283
Recall: 1.0
F0.5-score: 0.2699
Profit: -1670

Эпоха 2 | Train Loss: 0.8362 | Val ROC-AUC: 0.9596 | Val Profit: -1670
model_8_weight_decay | valid epoch 3
ROC-AUC: 0.9736
Precision: 0.3182
Recall: 1.0
F0.5-score: 0.3684
Profit: -1020

Эпоха 3 | Train Loss: 0.5543 | Val ROC-AUC: 0.9736 | Val Profit: -1020
model_8_weight_decay | valid epoch 4
ROC-AUC: 0.96
Precision: 0.2838
Recall: 1.0
F0.5-score: 0.3312
Profit: -1220

Эпоха 4 | Train Loss: 0.4349 | Val ROC-AUC: 0.9600 | Val Profit: -1220
model_8_weight_decay | valid epoch 5
ROC-AUC: 0.9583
Precision: 0.2632
Recall: 0.9524
F0.5-score: 0.3077
Profit: -1305

Эпоха 5 | Train Loss: 0.4016 | Val ROC-AUC: 0.9583 | Val Profit: -1305
model_8_weight_decay | valid epoch 6
ROC-AUC: 0.9671
Precis

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_8_weight_decay | valid epoch 34
ROC-AUC: 0.9722
Precision: 0.5294
Recall: 0.8571
F0.5-score: 0.5732
Profit: -325

Эпоха 34 | Train Loss: 0.2438 | Val ROC-AUC: 0.9722 | Val Profit: -325
model_8_weight_decay | valid epoch 35
ROC-AUC: 0.9838
Precision: 0.5926
Recall: 0.7619
F0.5-score: 0.6202
Profit: -220

Эпоха 35 | Train Loss: 0.2121 | Val ROC-AUC: 0.9838 | Val Profit: -220
model_8_weight_decay | valid epoch 36
ROC-AUC: 0.985
Precision: 0.9
Recall: 0.4286
F0.5-score: 0.7377
Profit: -40

Эпоха 36 | Train Loss: 0.1679 | Val ROC-AUC: 0.9850 | Val Profit: -40
model_8_weight_decay | valid epoch 37
ROC-AUC: 0.939
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 37 | Train Loss: 0.1302 | Val ROC-AUC: 0.9390 | Val Profit: -105
model_8_weight_decay | valid epoch 38
ROC-AUC: 0.8451
Precision: 0.6667
Recall: 0.0952
F0.5-score: 0.303
Profit: -110

Эпоха 38 | Train Loss: 0.1204 | Val ROC-AUC: 0.8451 | Val Profit: -110


/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_8_weight_decay | valid epoch 39
ROC-AUC: 0.9653
Precision: 0.2692
Recall: 1.0
F0.5-score: 0.3153
Profit: -1320

Эпоха 39 | Train Loss: 0.2697 | Val ROC-AUC: 0.9653 | Val Profit: -1320
model_8_weight_decay | valid epoch 40
ROC-AUC: 0.981
Precision: 0.5926
Recall: 0.7619
F0.5-score: 0.6202
Profit: -220

Эпоха 40 | Train Loss: 0.2274 | Val ROC-AUC: 0.9810 | Val Profit: -220

Лучшая эпоха для model_8_weight_decay: 36
Лучший ROC-AUC: 0.9849765258215962
Train
ROC-AUC: 0.9922
Precision: 0.9032
Recall: 0.3373
F0.5-score: 0.6763
Profit: -210

Test
ROC-AUC: 0.9654
Precision: 0.8359
Recall: 0.3623
F0.5-score: 0.6627
Profit: -74865



model_8_weight_decay/train_loss,█▆▄▃▃▃▂▂▂▂▁▂▂▂▂▂▂▂▂▂▂▁▁▂▂▂▂▁▁▁▁▁▂▂▂▁▁▁▂▂
model_8_weight_decay/valid_f05,▃▄▄▄▄▄▅▅▅▆▇▇▆▅▇▇▇▆▄▆▇▅▅▄▄▆▆▆▇▇▁▇▆▆▇█▁▄▄▇
model_8_weight_decay/valid_precision,▂▃▃▃▃▃▄▄▄▅▆▆▅▄▆▅▆▄▃▅▆▆▇▃▃▅▅▅▆▆▁▅▅▅▆█▁▆▃▆
model_8_weight_decay/valid_profit,▁▃▅▄▄▅▆▆▅▇██▇▆▇▇█▆▄▇███▄▅▇▇▇███▇▇▇▇███▄▇
model_8_weight_decay/valid_recall,██████▇▆█▆▅▆▇█▆▆▄██▇▆▂▂██▇▅▆▄▅▁▇▆▇▆▄▁▂█▆
model_8_weight_decay/valid_roc_auc,▆▇▇▇▇▇▇▇▇▇▇▇█▇▇▇██▇███▄████▇▆█▁█▇▇██▆▂▇█
model_8_weight_decay/train_loss,0.22739
model_8_weight_decay/valid_f05,0.62016
model_8_weight_decay/valid_precision,0.59259
model_8_weight_decay/valid_profit,-220
model_8_weight_decay/valid_recall,0.7619


In [142]:
train_metrics = evaluate_model(model_8, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_8, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9922
Precision: 0.9032
Recall: 0.3373
F0.5-score: 0.6763
Profit: -210

Test
ROC-AUC: 0.9654
Precision: 0.8359
Recall: 0.3623
F0.5-score: 0.6627
Profit: -74865



Добавив регуляризацию видим, что качество улучшилось относительно бейзлайна, практически на уровне чистого batchnorm

При этом пришлось обратно вернуться к обычной RELU

## model_9_Focal_loss

In [143]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        targets = targets.float()

        bce_loss = nn.functional.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none"
        )

        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        focal_weight = self.alpha * (1 - pt) ** self.gamma
        loss = focal_weight * bce_loss

        return loss.mean()

In [144]:
EPOCHS = 30
SEED = 42

In [145]:
wandb.init(mode="disabled")
results = []
for LR in [0.01, 0.005]:
    for GAMMA in [1.0, 2.0, 3.0]:
        for ALPHA in [0.25, 0.5, 0.99]:
            for DROPOUT_COEF in [0.0, 0.1, 0.2]:
                for WEIGHT_DECAY in [0.0, 0.0001, 0.001]:
                    random.seed(SEED)
                    np.random.seed(SEED)
                    torch.manual_seed(SEED) 

                    model_9 = nn.Sequential(
                        nn.Linear(9, 128),
                        nn.BatchNorm1d(128),
                        nn.ReLU(),
                        nn.Dropout(DROPOUT_COEF),

                        nn.Linear(128, 64),
                        nn.BatchNorm1d(64),
                        nn.ReLU(),
                        nn.Dropout(DROPOUT_COEF),

                        nn.Linear(64, 32),
                        nn.BatchNorm1d(32),
                        nn.ReLU(),
                        nn.Dropout(DROPOUT_COEF),

                        nn.Linear(32, 16),
                        nn.BatchNorm1d(16),
                        nn.ReLU(),
                        nn.Dropout(DROPOUT_COEF),

                        nn.Linear(16, 8),
                        nn.BatchNorm1d(8),
                        nn.ReLU(),
                        nn.Dropout(DROPOUT_COEF),

                        nn.Linear(8, 1)
                    )

                    loss_fn = FocalLoss(alpha=ALPHA, gamma=GAMMA)
                    optimizer = torch.optim.Adam(model_9.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
                    model_9 = train_model(
                        model=model_9,
                        train_loader=train_loader,
                        X_valid=X_val,
                        y_valid=y_val,
                        loss_fn=loss_fn,
                        optimizer=optimizer,
                        epochs=EPOCHS,
                        threshold=0.5,
                        model_name="model_9_Focal_loss"
                    )
                    val = evaluate_model(model_9, X_val, y_val, threshold=0.5, name="val")
                    results.append({"lr": LR, "gamma": GAMMA, "alpha": ALPHA, "dropout": DROPOUT_COEF, "weight_decay": WEIGHT_DECAY, "profit": val["profit"], "roc_auc": val["roc_auc"]})

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_9_Focal_loss | valid epoch 1
ROC-AUC: 0.9077
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 1 | Train Loss: 0.0425 | Val ROC-AUC: 0.9077 | Val Profit: -105
model_9_Focal_loss | valid epoch 2
ROC-AUC: 0.9442
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 2 | Train Loss: 0.0178 | Val ROC-AUC: 0.9442 | Val Profit: -105
model_9_Focal_loss | valid epoch 3
ROC-AUC: 0.9665
Precision: 0.3953
Recall: 0.8095
F0.5-score: 0.4404
Profit: -585

Эпоха 3 | Train Loss: 0.0136 | Val ROC-AUC: 0.9665 | Val Profit: -585
model_9_Focal_loss | valid epoch 4
ROC-AUC: 0.9685
Precision: 0.4483
Recall: 0.619
F0.5-score: 0.4745
Profit: -375

Эпоха 4 | Train Loss: 0.0118 | Val ROC-AUC: 0.9685 | Val Profit: -375
model_9_Focal_loss | valid epoch 5
ROC-AUC: 0.9639
Precision: 0.5
Recall: 0.7619
F0.5-score: 0.5369
Profit: -345

Эпоха 5 | Train Loss: 0.0114 | Val ROC-AUC: 0.9639 | Val Profit: -345
model_9_Focal_loss | valid epoch 6
ROC-AUC: 0.9722
Precision: 0.5357
Recall: 0.7143
F0.

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

model_9_Focal_loss | valid epoch 2
ROC-AUC: 0.9427
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 2 | Train Loss: 0.0177 | Val ROC-AUC: 0.9427 | Val Profit: -105
model_9_Focal_loss | valid epoch 3
ROC-AUC: 0.959
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 3 | Train Loss: 0.0147 | Val ROC-AUC: 0.9590 | Val Profit: -105
model_9_Focal_loss | valid epoch 4
ROC-AUC: 0.9439
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 4 | Train Loss: 0.0145 | Val ROC-AUC: 0.9439 | Val Profit: -105
model_9_Focal_loss | valid epoch 5
ROC-AUC: 0.9725
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 5 | Train Loss: 0.0134 | Val ROC-AUC: 0.9725 | Val Profit: -105


/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

...

model_9_Focal_loss | valid epoch 1
ROC-AUC: 0.8909
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 1 | Train Loss: 0.0486 | Val ROC-AUC: 0.8909 | Val Profit: -105
model_9_Focal_loss | valid epoch 2
ROC-AUC: 0.9362
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 2 | Train Loss: 0.0256 | Val ROC-AUC: 0.9362 | Val Profit: -105
model_9_Focal_loss | valid epoch 3
ROC-AUC: 0.9297
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 3 | Train Loss: 0.0176 | Val ROC-AUC: 0.9297 | Val Profit: -105
model_9_Focal_loss | valid epoch 4
ROC-AUC: 0.9465
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 4 | Train Loss: 0.0177 | Val ROC-AUC: 0.9465 | Val Profit: -105
model_9_Focal_loss | valid epoch 5
ROC-AUC: 0.9526
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 5 | Train Loss: 0.0148 | Val ROC-AUC: 0.9526 | Val Profit: -105
model_9_Focal_loss | valid epoch 6
ROC-AUC: 0.9606
Precision: 0.6667
Recall: 0.1905
F0.5-score: 0.4444
Profit:

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

model_9_Focal_loss | valid epoch 11
ROC-AUC: 0.9712
Precision: 0.7
Recall: 0.3333
F0.5-score: 0.5738
Profit: -110

Эпоха 11 | Train Loss: 0.0072 | Val ROC-AUC: 0.9712 | Val Profit: -110
model_9_Focal_loss | valid epoch 12
ROC-AUC: 0.9714
Precision: 0.5789
Recall: 0.5238
F0.5-score: 0.567
Profit: -195

Эпоха 12 | Train Loss: 0.0079 | Val ROC-AUC: 0.9714 | Val Profit: -195
model_9_Focal_loss | valid epoch 13
ROC-AUC: 0.9712
Precision: 0.5714
Recall: 0.5714
F0.5-score: 0.5714
Profit: -210

Эпоха 13 | Train Loss: 0.0121 | Val ROC-AUC: 0.9712 | Val Profit: -210
model_9_Focal_loss | valid epoch 14
ROC-AUC: 0.9627
Precision: 0.4286
Recall: 0.1429
F0.5-score: 0.3061
Profit: -175

Эпоха 14 | Train Loss: 0.0103 | Val ROC-AUC: 0.9627 | Val Profit: -175
model_9_Focal_loss | valid epoch 15
ROC-AUC: 0.8703
Precision: 0.3333
Recall: 0.0476
F0.5-score: 0.1515
Profit: -145

Эпоха 15 | Train Loss: 0.0084 | Val ROC-AUC: 0.8703 | Val Profit: -145
model_9_Focal_loss | valid epoch 16
ROC-AUC: 0.9199
Precisi

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_9_Focal_loss | valid epoch 25
ROC-AUC: 0.9337
Precision: 0.7143
Recall: 0.2381
F0.5-score: 0.5102
Profit: -105

Эпоха 25 | Train Loss: 0.0033 | Val ROC-AUC: 0.9337 | Val Profit: -105
model_9_Focal_loss | valid epoch 26
ROC-AUC: 0.8873
Precision: 1.0
Recall: 0.0952
F0.5-score: 0.3448
Profit: -85

Эпоха 26 | Train Loss: 0.0033 | Val ROC-AUC: 0.8873 | Val Profit: -85
model_9_Focal_loss | valid epoch 27
ROC-AUC: 0.9769
Precision: 0.5769
Recall: 0.7143
F0.5-score: 0.6
Profit: -230

Эпоха 27 | Train Loss: 0.0038 | Val ROC-AUC: 0.9769 | Val Profit: -230
model_9_Focal_loss | valid epoch 28
ROC-AUC: 0.9636
Precision: 0.4167
Recall: 0.9524
F0.5-score: 0.4695
Profit: -605

Эпоха 28 | Train Loss: 0.0077 | Val ROC-AUC: 0.9636 | Val Profit: -605
model_9_Focal_loss | valid epoch 29
ROC-AUC: 0.9737
Precision: 0.4634
Recall: 0.9048
F0.5-score: 0.5135
Profit: -465

Эпоха 29 | Train Loss: 0.0099 | Val ROC-AUC: 0.9737 | Val Profit: -465
model_9_Focal_loss | valid epoch 30
ROC-AUC: 0.9752
Precision: 

In [146]:
best = max(results, key=lambda r: r["profit"])
print("Лучшие гиперпараметры:", best)

Лучшие гиперпараметры: {'lr': 0.005, 'gamma': 2.0, 'alpha': 0.99, 'dropout': 0.0, 'weight_decay': 0.0, 'profit': np.int64(-10), 'roc_auc': 0.9880617035546613}


In [147]:
LR = best["lr"]
GAMMA = best["gamma"]
ALPHA = best["alpha"]
DROPOUT_COEF = best["dropout"]
WEIGHT_DECAY = best["weight_decay"]

In [148]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED) 


model_9 = nn.Sequential(
    nn.Linear(9, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(64, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(32, 16),
    nn.BatchNorm1d(16),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(16, 8),
    nn.BatchNorm1d(8),
    nn.ReLU(),
    nn.Dropout(DROPOUT_COEF),

    nn.Linear(8, 1)
)

loss_fn = FocalLoss(alpha=ALPHA, gamma=GAMMA)
optimizer = torch.optim.Adam(model_9.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

config = {
    "model": "MLP_focal_loss",
    "optimizer": str(optimizer.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "alpha": ALPHA,
    "gamma": GAMMA,
    "pos_weight": pos_weight,
    "loss": str(loss_fn.__class__.__name__),
    "architecture": str(model_9)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="model_9_focal_loss", config=config)
log_file = new_log_file()

model_9 = train_model(
    model=model_9,
    train_loader=train_loader,
    X_valid=X_val,
    y_valid=y_val,
    loss_fn=loss_fn,
    optimizer=optimizer,
    epochs=EPOCHS,
    threshold=0.5,
    model_name="model_9_Focal_loss"
)


save_results(model_9, "model_9", log_file, run)

run.finish()

/Users/katya/Desktop/Катя/ВШЭ/смадимо/gp-5/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


model_9_Focal_loss | valid epoch 1
ROC-AUC: 0.9249
Precision: 0.0
Recall: 0.0
F0.5-score: 0.0
Profit: -105

Эпоха 1 | Train Loss: 0.0929 | Val ROC-AUC: 0.9249 | Val Profit: -105
model_9_Focal_loss | valid epoch 2
ROC-AUC: 0.9575
Precision: 0.5
Recall: 0.4762
F0.5-score: 0.495
Profit: -255

Эпоха 2 | Train Loss: 0.0445 | Val ROC-AUC: 0.9575 | Val Profit: -255
model_9_Focal_loss | valid epoch 3
ROC-AUC: 0.9549
Precision: 0.4242
Recall: 0.6667
F0.5-score: 0.4575
Profit: -440

Эпоха 3 | Train Loss: 0.0319 | Val ROC-AUC: 0.9549 | Val Profit: -440
model_9_Focal_loss | valid epoch 4
ROC-AUC: 0.9581
Precision: 0.48
Recall: 0.5714
F0.5-score: 0.4959
Profit: -310

Эпоха 4 | Train Loss: 0.0254 | Val ROC-AUC: 0.9581 | Val Profit: -310
model_9_Focal_loss | valid epoch 5
ROC-AUC: 0.969
Precision: 0.4737
Recall: 0.8571
F0.5-score: 0.5202
Profit: -425

Эпоха 5 | Train Loss: 0.0229 | Val ROC-AUC: 0.9690 | Val Profit: -425
model_9_Focal_loss | valid epoch 6
ROC-AUC: 0.971
Precision: 0.5652
Recall: 0.619

model_9_Focal_loss/train_loss,█▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂▁▁▁▁▁▂▁▁▁
model_9_Focal_loss/valid_f05,▁▅▅▅▅▆▆▅▆▅▆▅▆▇█▅▄▃▅▅▄▅▅▆▇▆▆▁▃▅
model_9_Focal_loss/valid_precision,▁▅▄▅▅▆▆▆▆▅▆▆▇▆█▅▅▅▅▆▆▆▆▆▇▆▇▁▅▆
model_9_Focal_loss/valid_profit,▆▄▁▃▁▅▅▆▅▂▆▅▆▆█▅▅▆▃▆▆▆▆▆▇▆▆▆▆▆
model_9_Focal_loss/valid_recall,▁▅▆▆█▆▆▃▆▇▅▅▄▇█▄▂▂▅▃▂▃▃▅▅▅▅▁▂▃
model_9_Focal_loss/valid_roc_auc,▁▅▄▅▆▆▆▇▇▅▆▅▆▇█▆▇▄▅▆▇▆▃▇▂▄▆▅▅▂
model_9_Focal_loss/train_loss,0.00327
model_9_Focal_loss/valid_f05,0.44444
model_9_Focal_loss/valid_precision,0.66667
model_9_Focal_loss/valid_profit,-115
model_9_Focal_loss/valid_recall,0.19048


In [149]:
train_metrics = evaluate_model(model_9, X_train, y_train, threshold=0.5, name="Train")
test_metrics = evaluate_model(model_9, X_test, y_test, threshold=0.5, name="Test")

Train
ROC-AUC: 0.9916
Precision: 0.831
Recall: 0.7108
F0.5-score: 0.8038
Profit: -125

Test
ROC-AUC: 0.9702
Precision: 0.6867
Recall: 0.5698
F0.5-score: 0.6597
Profit: -137610



В этом эксперименте пробовали ментять функцию активации, но это особо не помогло. Использовал FocalLoss. И перебирали параметры

На валидации profit отличный, но на тесте хуже бейзлайна. Такой большой разрыв говорит о переобучении


# ансамбль ПОКА НЕ ТРОГАЛ

In [ ]:
import numpy as np

In [ ]:
def build_model():
    return nn.Sequential(
        nn.Linear(9, 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(64, 32),
        nn.BatchNorm1d(32),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(32, 1),
        )

In [ ]:
EPOCHS = 120
LR = 0.001
WEIGHT_DECAY = 1e-4
N_ENSEMBLE = 5

net = build_model()
opt = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

In [ ]:
loss_fn2 = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [ ]:
config = {
    "model": "Ensemble",
    "optimizer": str(opt.__class__.__name__),
    "task": "fraud_detection",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "pos_weight": pos_weight,
    "loss": str(loss_fn2.__class__.__name__),
    "n_ensemble": N_ENSEMBLE,
    "architecture": str(net)
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="ensemble", config=config)

In [ ]:
test_probs = []
logging.info("Запустили обучение моделей ансамбля")

np.random.seed(42)
seeds = np.random.randint(1, 525252, size=N_ENSEMBLE)
ensemble = []
for i in range(N_ENSEMBLE):
    seed = seeds[i]
    torch.manual_seed(seed)
    net = build_model()
    opt = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    logging.info(f"Запустили обучение модели {i+1}. Сид: {seed}")


    net.train()
    for epoch in range(1, EPOCHS + 1):
        epoch_loss = 0
        for xb, yb in train_loader:
            opt.zero_grad()
            loss = loss_fn2(net(xb), yb)
            loss.backward()
            epoch_loss += loss.item()
            opt.step()
        
        logging.info(f"эпоха: {epoch}, loss:  {round(epoch_loss, 4)}")
        wandb.log({"epoch": epoch, f"ensemble/{i}/train_loss": epoch_loss})
    
    ensemble.append(net.state_dict())

    net.eval()
    with torch.no_grad():
        test_probs.append(torch.sigmoid(net(X_test)).numpy())

    logging.info(f"Сеть {i} обучена")
    print("сеть", i, "обучена")

probs2 = np.mean(test_probs, axis=0)

In [ ]:
auc_score = roc_auc_score(y_true, probs2)
logging.info(f"На тесте ROC_AUC: {auc_score}")
auc_score

In [ ]:
pre_score = average_precision_score(y_true, probs2)
logging.info(f"На тесте Precision: {pre_score}")
pre_score

In [ ]:
results = wandb.Table(columns=['model', 'test/avg_roc_auc', 'test/avg_precision'])
results.add_data("ensemble", auc_score, pre_score)
wandb.log({"results": results})

In [ ]:
import pickle

pickle.dump(ensemble, open("models/model_fraud_ensemble.pkl", 'wb'))
logging.info("Сохранили веса моделей ансамбля в папку models")

In [ ]:
artifact = wandb.Artifact(name="model_fraud_ensemble.pkl", type="model", description="Ансамбль моделей для определения Фрода")

artifact.add_file("models/model_fraud_ensemble.pkl")
artifact.add_file(log_file)
run.log_artifact(artifact)


In [ ]:
run.finish()

In [36]:
wandb.finish()


In [608]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix
import numpy as np

k_values = [3, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15]

best_k = None
best_roc_auc = -1
best_knn = None

X_train_np = X_train.numpy()
y_train_np = y_train.numpy().ravel()

X_val_np = X_val.numpy()
y_val_np = y_val.numpy().ravel()

X_test_np = X_test.numpy()
y_test_np = y_test.numpy().ravel()

for k in k_values:
    knn = KNeighborsClassifier(
        n_neighbors=k,
        weights="distance"
    )

    knn.fit(X_train_np, y_train_np)

    val_probs = knn.predict_proba(X_val_np)[:, 1]
    val_roc_auc = roc_auc_score(y_val_np, val_probs)

    print(f"k={k} | val ROC-AUC={val_roc_auc:.4f}")

    if val_roc_auc > best_roc_auc:
        best_roc_auc = val_roc_auc
        best_k = k
        best_knn = knn

print("\nЛучший KNN")
print("-" * 30)
print("best k:", best_k)
print("best val ROC-AUC:", round(best_roc_auc, 4))

k=3 | val ROC-AUC=0.7783
k=5 | val ROC-AUC=0.8613
k=7 | val ROC-AUC=0.8891
k=8 | val ROC-AUC=0.9312
k=9 | val ROC-AUC=0.9356
k=10 | val ROC-AUC=0.9371
k=11 | val ROC-AUC=0.9336
k=12 | val ROC-AUC=0.9298
k=13 | val ROC-AUC=0.9272
k=14 | val ROC-AUC=0.9328
k=15 | val ROC-AUC=0.9313

Лучший KNN
------------------------------
best k: 10
best val ROC-AUC: 0.9371


In [609]:
threshold = 0.5

test_probs = best_knn.predict_proba(X_test_np)[:, 1]
test_pred = (test_probs >= threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test_np, test_pred).ravel()

test_profit = tp * 5 - fp * 25 - fn * 5
test_roc_auc = roc_auc_score(y_test_np, test_probs)

print("\nKNN на test")
print("-" * 30)
print("k:", best_k)
print("threshold:", threshold)
print("ROC-AUC:", round(test_roc_auc, 4))
print("Profit:", test_profit)
print("TN, FP, FN, TP:", tn, fp, fn, tp)


KNN на test
------------------------------
k: 10
threshold: 0.5
ROC-AUC: 0.8991
Profit: -130230
TN, FP, FN, TP: 473649 745 23024 703
